In [1]:
import os
import sys

project_root = "/root/work/tvm-ansor"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build-release"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build-release")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build-release:" + os.environ.get("LD_LIBRARY_PATH", "")


import numpy as np
from util_manager import PathManager, get_network
import tvm
from tvm import auto_scheduler

TARGET = tvm.target.Target("cuda")

def get_tasks(mod, params, path_manager, verbose=True, get_pkl=True):
    if get_pkl:
        tasks, task_weights = path_manager.tasks_pkl_use()
    
    if get_pkl is False or tasks is None:
        print("Extract tasks...")
        tasks, task_weights = auto_scheduler.extract_tasks(mod["main"], params, TARGET)
        if path_manager.tasks_pkl_check() is False:
            path_manager.tasks_pkl_save(tasks, task_weights)

    if verbose:
        for idx, task in enumerate(tasks):
            print("========== Task %d : %s  (workload key: %s) ==========" % (idx, task.desc, task.workload_key))
            print(task.compute_dag)
    
    print(f"Total tasks length : {len(tasks)}")
    return tasks, task_weights


In [2]:
from types import SimpleNamespace

args = SimpleNamespace(
    network="resnet_18",
    batch_size=1,
    dtype="float32",
    layout="NHWC",
    timenow=None,
    json=None
)

mod, params, input_shape, output_shape = get_network(args.network, args.batch_size, args.layout, dtype=args.dtype)
path_manager = PathManager(args.network, input_shape, args, None, json="/root/work/tvm-ansor/gallery/logs_json/tmp.json")
tasks, task_weights = get_tasks(None, params, path_manager, verbose=False, get_pkl=True)
tasks, task_weights = zip(*sorted(zip(tasks, task_weights), key=lambda x: x[0].desc))

search_policies = []
for idx, (task, weight) in enumerate(zip(tasks, task_weights)):
    print(f"T{idx} : {task.desc} ({weight})")
    search_policies.append(
        auto_scheduler.SketchPolicy(task, auto_scheduler.XGBModel())
    )

Getting network resnet_18...

Using json : /root/work/tvm-ansor/gallery/logs_json/tmp.json
Load tasks from /root/work/tvm-ansor/gallery/ansor_tasks_pkl/resnet_18-(1,224,224,3).pkl
Total tasks length : 24
T0 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add (1)
T1 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_1 (1)
T2 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_2 (1)
T3 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_3 (1)
T4 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_add_nn_relu (1)
T5 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_add_nn_relu_1 (1)
T6 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_add_nn_relu_2 (1)
T7 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_multiply_add_nn_re_a4e88f85cf7823fc_ (1)
T8 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu (2)
T9 

In [100]:
builtins_min = min

"""
SymbolicState: task.compute_dag의 stage/iter 구조를 그대로 복사하고,
transform step 적용 시 split/unroll factor를 symbolic variable로 표현하는 객체.

C++ loop_state.cc의 State/Stage/Iterator 구조를 Python으로 미러링합니다.

변수 네이밍:
  - SplitStep:             sp_{step_idx}_{length_idx}
  - FollowSplitStep:       src SplitStep의 sym_name 재사용
  - FollowFusedSplitStep:  src SplitStep들의 sym_name 곱
  - PragmaStep (unroll):   ur_{step_idx}

ComputeAt extent 복원 전략 (lazy):
  ComputeAt는 extent를 None으로만 설정.
  이후 Fuse/Split/FFSP에서 None extent를 만나면 그 시점(step_idx)에서
  ReplayStepsPartial + InferBound를 호출하여 concrete extent를 동적 복원.
  이렇게 해야 부모 stage의 split 결과가 반영된 정확한 extent를 얻을 수 있음.
"""
from collections import OrderedDict
import math

# ── Annotation 문자열 매핑 (C++ IteratorAnnotationString 동일) ──
ANNOTATION_STR = {
    0: "for",           # kNone
    1: "unroll",        # kUnroll
    2: "vectorize",     # kVectorize
    3: "parallel",      # kParallel
    4: "vthread",       # kVThread
    5: "blockIdx.x",    # kBlockX
    6: "threadIdx.x",   # kThreadX
    7: "blockIdx.y",    # kBlockY
    8: "threadIdx.y",   # kThreadY
    9: "blockIdx.z",    # kBlockZ
    10: "threadIdx.z",  # kThreadZ
    11: "tensorize",    # kTensorize
}

# ── ComputeAtKind ──
CA_ROOT    = 0  # kRoot
CA_INLINED = 1  # kInlined
CA_ITER    = 2  # kIter


class SymExpr:
    """
    Symbolic expression wrapper.
    실제 값(int)이면 그냥 int, symbolic이면 문자열.
    연산(ceil div, mul 등)을 문자열로 합성.
    """
    def __init__(self, val):
        self.val = val

    @property
    def is_concrete(self):
        return isinstance(self.val, int)

    def __repr__(self):
        return str(self.val) if self.val is not None else "None"

    def __str__(self):
        return str(self.val) if self.val is not None else "None"

    def __int__(self):
        if self.is_concrete:
            return self.val
        raise ValueError(f"Cannot convert symbolic '{self.val}' to int")

    @staticmethod
    def ceildiv(a, b):
        if isinstance(a, SymExpr): a = a.val
        if isinstance(b, SymExpr): b = b.val
        if isinstance(a, int) and isinstance(b, int):
            return SymExpr((a + b - 1) // b)
        return SymExpr(f"ceil({a}/({b}))")

    @staticmethod
    def _needs_parens_for_mul(s):
        """문자열 expression이 mul에서 사용될 때 괄호가 필요한지 판단.
        최외곽 레벨에서 + 또는 - 연산자가 있으면 True."""
        if not isinstance(s, str):
            return False
        depth = 0
        i = 0
        while i < len(s):
            c = s[i]
            if c == '(':
                depth += 1
            elif c == ')':
                depth -= 1
            elif depth == 0 and c == '+':
                return True
            elif depth == 0 and c == '-' and i > 0 and s[i-1] == ' ':
                return True
            i += 1
        return False

    @staticmethod
    def mul(a, b):
        if isinstance(a, SymExpr): a = a.val
        if isinstance(b, SymExpr): b = b.val
        if isinstance(a, int) and isinstance(b, int):
            return SymExpr(a * b)
        if a == 1: return SymExpr(b)
        if b == 1: return SymExpr(a)
        # 연산자 우선순위 보호: 최외곽에 +/- 포함된 표현식은 괄호 추가
        a_str = str(a)
        b_str = str(b)
        if SymExpr._needs_parens_for_mul(a_str):
            a_str = f"({a_str})"
        if SymExpr._needs_parens_for_mul(b_str):
            b_str = f"({b_str})"
        return SymExpr(f"{a_str}*{b_str}")

    @staticmethod
    def product(items):
        result = SymExpr(1)
        for item in items:
            result = SymExpr.mul(result, item)
        return result

    @staticmethod
    def min(a, b):
        """min(a, b) — both concrete → int min, otherwise symbolic min string."""
        if a is None or b is None:
            return a if b is None else b
        if isinstance(a, SymExpr): a_val = a.val
        else: a_val = a
        if isinstance(b, SymExpr): b_val = b.val
        else: b_val = b
        if isinstance(a_val, int) and isinstance(b_val, int):
            return SymExpr(builtins_min(a_val, b_val))
        return SymExpr(f"min({a_val},{b_val})")


class SymIter:
    """Iterator (C++ Iterator 대응)"""
    def __init__(self, name, extent, annotation=0, iter_kind=0):
        self.name = name
        self.extent = extent         # SymExpr or None
        self.annotation = annotation # int
        self.iter_kind = iter_kind   # int

    def clone(self):
        return SymIter(self.name,
                       SymExpr(self.extent.val) if self.extent else None,
                       self.annotation, self.iter_kind)

    def __repr__(self):
        ann = ANNOTATION_STR.get(self.annotation, "?")
        if self.extent is not None:
            return f"{ann} {self.name} (0,{self.extent})"
        else:
            return f"{ann} {self.name} (None)"


class SymStage:
    """Stage (C++ Stage 대응)"""
    def __init__(self, op_name, op_type, iters, compute_at=CA_ROOT,
                 auto_unroll_max_step=None, storage_offset=0, dtype="float32"):
        self.op_name = op_name
        self.op_type = op_type
        self.iters = list(iters)
        self.compute_at = compute_at
        self.auto_unroll_max_step = auto_unroll_max_step
        self.storage_offset = storage_offset
        self.attach_stage_id = None
        self.attach_iter_id = None
        self.dtype = dtype  # e.g. "float32", "float16", "int8"

    @property
    def dtype_bytes(self):
        """dtype의 바이트 수를 반환."""
        import tvm
        return tvm.DataType(self.dtype).bits // 8


def eval_sym_extent(expr, sym_map):
    """SymExpr의 문자열을 sym_map으로 치환하여 eval로 계산"""
    if expr is None:
        return None
    s_val = str(expr)
    if s_val == "None":
        return None
    try:
        return int(s_val)
    except ValueError:
        pass
    evaluated = s_val
    evaluated = evaluated.replace("ceil(", "math.ceil(")
    for sym_name in sorted(sym_map.keys(), key=len, reverse=True):
        if sym_map[sym_name] is not None:
            evaluated = evaluated.replace(sym_name, str(sym_map[sym_name]))
    try:
        return int(eval(evaluated))
    except Exception:
        return f"EVAL_FAIL({s_val}→{evaluated})"


In [101]:
# ═══════════════════════════════════════════════════════════════
# SymbolicState  – 순수 상태 (stages, sym_map, 출력, extent 조회)
# TransformApplier – transform step 적용 로직
# SymParamManager – 파라미터 조회 / 검증 / 랜덤 생성
# ═══════════════════════════════════════════════════════════════


# ─────────────────────────────────────────────────────────────
#  SymbolicState: 순수 상태 객체
# ─────────────────────────────────────────────────────────────
class SymbolicState:
    """
    Symbolic 버전의 auto_scheduler State.
    stages, sym_map, 내부 메타데이터를 보유하는 순수 상태 객체.
    transform step 적용은 TransformApplier, 파라미터 관리는 SymParamManager가 담당.
    """

    @staticmethod
    def _safe_int_extent(extent_expr):
        """TIR extent를 int로 변환. Sub/Add 등 심볼릭이면 simplify 후 재시도."""
        if extent_expr is None:
            return None
        try:
            return int(extent_expr)
        except TypeError:
            import tvm
            simplified = tvm.arith.Analyzer().simplify(extent_expr)
            return int(simplified)

    def __init__(self, compute_dag):
        self.stages = []
        self.sym_map = OrderedDict()
        self.compute_dag = compute_dag
        self._state = None  # TransformApplier.apply_steps에서 설정
        self._ca_saved_extents = {}  # {(stage_id, iter_id): SymExpr}
        self._split_sym_products = {}  # {(stage_id, step_idx): SymExpr}
        self._cache_read_consumer = {}  # {cache_read_stage_id: consumer_stage_id}
        self._cache_read_stencil_info = {}  # {cr_stage_id: {cr_axis_idx: (stride, sp_order, rd_order)}}
        self._shared_fused_extents = {}  # {stage_id: SymExpr}

        for sid, op in enumerate(compute_dag.ops):
            if hasattr(op, 'axis'):
                dtype = str(op.output(0).dtype) if hasattr(op, 'output') else "float32"
                iters = []
                for axis in op.axis:
                    name = str(axis.var.name)
                    ext = self._safe_int_extent(axis.dom.extent) if axis.dom is not None else None
                    iters.append(SymIter(name, SymExpr(ext) if ext is not None else None,
                                         annotation=0, iter_kind=0))
                for axis in op.reduce_axis:
                    name = str(axis.var.name)
                    ext = self._safe_int_extent(axis.dom.extent) if axis.dom is not None else None
                    iters.append(SymIter(name, SymExpr(ext) if ext is not None else None,
                                         annotation=0, iter_kind=1))
                self.stages.append(SymStage(op.name, 'compute', iters, dtype=dtype))
            else:
                dtype = str(op.output(0).dtype) if hasattr(op, 'output') else "float32"
                self.stages.append(SymStage(op.name, 'placeholder', [], dtype=dtype))

    # ─── 내부 데이터 shift (CacheRead/CacheWrite stage 삽입 시) ───
    def _shift_ca_saved_extents(self, inserted_stage_id):
        """CacheRead/CacheWrite로 stage가 삽입될 때, saved extents와 관련 데이터 key 업데이트."""
        new_saved = {}
        for (sid, iid), expr in self._ca_saved_extents.items():
            new_sid = sid + 1 if sid >= inserted_stage_id else sid
            new_saved[(new_sid, iid)] = expr
        self._ca_saved_extents = new_saved

        new_split_prods = {}
        for (sid, step_idx), expr in self._split_sym_products.items():
            new_sid = sid + 1 if sid >= inserted_stage_id else sid
            new_split_prods[(new_sid, step_idx)] = expr
        self._split_sym_products = new_split_prods

        new_cr_consumer = {}
        for cr_sid, consumer_sid in self._cache_read_consumer.items():
            new_cr = cr_sid + 1 if cr_sid >= inserted_stage_id else cr_sid
            new_con = consumer_sid + 1 if consumer_sid >= inserted_stage_id else consumer_sid
            new_cr_consumer[new_cr] = new_con
        self._cache_read_consumer = new_cr_consumer

        new_stencil = {}
        for cr_sid, info in self._cache_read_stencil_info.items():
            new_cr = cr_sid + 1 if cr_sid >= inserted_stage_id else cr_sid
            new_stencil[new_cr] = info
        self._cache_read_stencil_info = new_stencil

        new_shared = {}
        for sid, ext in self._shared_fused_extents.items():
            new_sid = sid + 1 if sid >= inserted_stage_id else sid
            new_shared[new_sid] = ext
        self._shared_fused_extents = new_shared

    # ─── 출력 ───
    def __str__(self):
        return self.to_str(delete_trivial_loop=False)

    def __repr__(self):
        return self.to_str(delete_trivial_loop=False)

    def to_str(self, delete_trivial_loop=True):
        lines = []
        placeholders = [s.op_name for s in self.stages if s.op_type == 'placeholder']
        lines.append("Placeholder: " + ", ".join(placeholders))
        for sid, stage in enumerate(self.stages):
            if stage.op_type == 'placeholder':
                continue
            if stage.compute_at == CA_ROOT:
                self._print_stage(lines, sid, 0, delete_trivial_loop)
        return "\n".join(lines)

    def _print_stage(self, lines, stage_id, base_indent, delete_trivial_loop):
        stage = self.stages[stage_id]
        if stage.auto_unroll_max_step is not None:
            lines.append(" " * base_indent + f"{stage.op_name} auto_unroll: {stage.auto_unroll_max_step}")
        if stage.storage_offset != 0:
            lines.append(" " * base_indent + f"{stage.op_name} storage_offset: {stage.storage_offset}")

        indent = 0
        for iid, it in enumerate(stage.iters):
            is_trivial = (it.extent is not None and it.extent.is_concrete and it.extent.val == 1)
            if not (delete_trivial_loop and is_trivial):
                ann = ANNOTATION_STR.get(it.annotation, "?")
                if it.extent is not None:
                    lines.append(" " * (base_indent + indent) + f"{ann} {it.name} (0,{it.extent})")
                else:
                    lines.append(" " * (base_indent + indent) + f"{ann} {it.name} (None)")
                indent += 2

            for asid, astage in enumerate(self.stages):
                if (astage.compute_at == CA_ITER and
                    astage.attach_stage_id == stage_id and
                    astage.attach_iter_id == iid):
                    self._print_stage(lines, asid, base_indent + indent, delete_trivial_loop)

        lines.append(" " * (base_indent + indent) + f"{stage.op_name} = ...")

    # ─── Symbolic extent 조회 함수 ───
    def _collect_extents_by_annotation(self, ann_codes):
        """주어진 annotation 코드 집합에 해당하는 iter의 (stage_id, iter_id, SymExpr) 목록 반환."""
        results = []
        for sid, stage in enumerate(self.stages):
            if stage.compute_at == CA_INLINED:
                continue
            for iid, it in enumerate(stage.iters):
                if it.annotation in ann_codes:
                    results.append((sid, iid, it.extent))
        return results

    def get_vectorize_extents(self):
        return self._collect_extents_by_annotation({2})

    def get_thread_extents(self):
        return self._collect_extents_by_annotation({6, 8, 10})

    def get_vthread_extents(self):
        return self._collect_extents_by_annotation({4})

    def get_shared_memory_extents(self):
        results = []
        for sid, ext in sorted(self._shared_fused_extents.items()):
            results.append((sid, self.stages[sid].op_name, ext))
        return results


# ─────────────────────────────────────────────────────────────
#  TransformApplier: transform step 적용 로직
# ─────────────────────────────────────────────────────────────
class TransformApplier:
    """
    SymbolicState에 transform steps를 순차 적용하는 로직.
    SymbolicState의 내부 데이터(stages, sym_map, 메타데이터)를 직접 조작.
    """

    def __init__(self, sym_state):
        """
        Args:
            sym_state: SymbolicState 인스턴스
        """
        self.s = sym_state  # SymbolicState 참조

    def apply_steps(self, state):
        """모든 transform steps를 순차 적용."""
        self.s._state = state
        steps = state.transform_steps
        for i, step in enumerate(steps):
            tk = step.type_key.split(".")[-1]
            if tk == "AnnotationStep":
                self._apply_annotation(step)
            elif tk == "FuseStep":
                self._apply_fuse(step, i)
            elif tk == "PragmaStep":
                self._apply_pragma(step, i)
            elif tk == "ReorderStep":
                self._apply_reorder(step)
            elif tk == "SplitStep":
                self._apply_split(step, i)
            elif tk == "FollowSplitStep":
                self._apply_follow_split(step, steps, i)
            elif tk == "FollowFusedSplitStep":
                self._apply_follow_fused_split(step, steps, i)
            elif tk == "StorageAlignStep":
                self._apply_storage_align(step)
            elif tk == "ComputeAtStep":
                self._apply_compute_at(step)
            elif tk == "ComputeInlineStep":
                self._apply_compute_inline(step)
            elif tk == "ComputeRootStep":
                self._apply_compute_root(step)
            elif tk == "CacheReadStep":
                self._apply_cache_read(step, state, i)
            elif tk == "CacheWriteStep":
                self._apply_cache_write(step, state, i)
            else:
                print(f"  [WARN] Unhandled step type: {tk}")

        self._infer_bound_final(state)

    # ─── Lazy extent 복원 ───
    def _restore_stage_extents_if_needed(self, stage_id, step_idx):
        s = self.s
        stage = s.stages[stage_id]
        has_none = any(it.extent is None for it in stage.iters)
        if not has_none:
            return

        ps = tvm._ffi.get_global_func("auto_scheduler.ReplayStepsPartial")(
            s.compute_dag, s._state, step_idx)
        bounded = s.compute_dag.infer_bound_from_state(ps)

        if stage_id >= len(bounded.stages):
            return

        cr_sym_candidates = None
        cr_stencil = None
        cr_ordered_splits = None
        if stage_id in s._cache_read_consumer:
            cr_sym_candidates = self._get_consumer_split_sym_products(stage_id)
            cr_stencil = s._cache_read_stencil_info.get(stage_id)
            if cr_stencil:
                consumer_sid = s._cache_read_consumer[stage_id]
                ordered = [(si, prod) for (sid, si), prod in s._split_sym_products.items()
                          if sid == consumer_sid]
                ordered.sort(key=lambda x: x[0])
                cr_ordered_splits = [(SymExpr(prod.val), eval_sym_extent(prod, s.sym_map))
                                    for si, prod in ordered]

        real_stage = bounded.stages[stage_id]
        for iid in range(len(stage.iters)):
            if stage.iters[iid].extent is None and iid < len(real_stage.iters):
                real_it = real_stage.iters[iid]
                if real_it.range is not None:
                    real_ext = int(real_it.range.extent)
                    saved = s._ca_saved_extents.get((stage_id, iid))
                    if saved is not None:
                        stage.iters[iid].extent = saved
                        continue
                    if cr_sym_candidates is not None:
                        if cr_stencil and cr_ordered_splits and iid in cr_stencil:
                            stride, sp_order, rd_order = cr_stencil[iid]
                            if stride == 0:
                                order = sp_order if sp_order is not None else rd_order
                                if order is not None and order < len(cr_ordered_splits):
                                    sym_expr, eval_val = cr_ordered_splits[order]
                                    stage.iters[iid].extent = sym_expr
                                    for ci_idx, (ev, se) in enumerate(cr_sym_candidates):
                                        if ev == eval_val and se.val == sym_expr.val:
                                            cr_sym_candidates.pop(ci_idx)
                                            break
                                    continue
                            elif stride > 0 and sp_order is not None and rd_order is not None:
                                sp_sym, sp_eval = cr_ordered_splits[sp_order]
                                rd_sym, rd_eval = cr_ordered_splits[rd_order]
                                predicted = (sp_eval - 1) * stride + rd_eval
                                if predicted == real_ext:
                                    stencil_expr = SymExpr(f"({sp_sym.val} - 1)*{stride} + {rd_sym.val}")
                                    stage.iters[iid].extent = stencil_expr
                                    continue
                        matched_sym = self._match_cr_extent(real_ext, cr_sym_candidates)
                        if matched_sym is not None:
                            stage.iters[iid].extent = matched_sym
                            cr_sym_candidates.remove((real_ext, matched_sym))
                            continue
                    stage.iters[iid].extent = SymExpr(real_ext)

    def _get_consumer_split_sym_products(self, cache_read_stage_id):
        s = self.s
        consumer_sid = s._cache_read_consumer.get(cache_read_stage_id)
        if consumer_sid is None:
            return None
        candidates = []
        for (sid, step_idx), sym_prod in s._split_sym_products.items():
            if sid == consumer_sid:
                eval_val = eval_sym_extent(sym_prod, s.sym_map)
                if isinstance(eval_val, int):
                    candidates.append((eval_val, SymExpr(sym_prod.val)))
        return candidates

    @staticmethod
    def _match_cr_extent(real_ext, candidates):
        for eval_val, sym_expr in candidates:
            if eval_val == real_ext:
                return sym_expr
        return None

    def _infer_bound_final(self, state):
        s = self.s
        from tvm.auto_scheduler.loop_state import StateObject
        state_obj = state if isinstance(state, StateObject) else state.state_object
        bounded = s.compute_dag.infer_bound_from_state(state_obj)

        for sid in range(len(s.stages)):
            sym_stage = s.stages[sid]
            if sid >= len(bounded.stages):
                continue
            real_stage = bounded.stages[sid]

            cr_sym_candidates = None
            cr_stencil = None
            cr_ordered_splits = None
            if sid in s._cache_read_consumer:
                cr_sym_candidates = self._get_consumer_split_sym_products(sid)
                cr_stencil = s._cache_read_stencil_info.get(sid)
                if cr_stencil:
                    consumer_sid = s._cache_read_consumer[sid]
                    ordered = [(si, prod) for (s2, si), prod in s._split_sym_products.items()
                              if s2 == consumer_sid]
                    ordered.sort(key=lambda x: x[0])
                    cr_ordered_splits = [(SymExpr(prod.val), eval_sym_extent(prod, s.sym_map))
                                        for si, prod in ordered]

            for iid in range(len(sym_stage.iters)):
                sym_it = sym_stage.iters[iid]
                if sym_it.extent is None and iid < len(real_stage.iters):
                    real_it = real_stage.iters[iid]
                    if real_it.range is None:
                        continue
                    real_ext = int(real_it.range.extent)
                    saved = s._ca_saved_extents.get((sid, iid))
                    if saved is not None:
                        sym_it.extent = saved
                        continue
                    if cr_sym_candidates is not None:
                        if cr_stencil and cr_ordered_splits and iid in cr_stencil:
                            stride, sp_order, rd_order = cr_stencil[iid]
                            if stride == 0:
                                order = sp_order if sp_order is not None else rd_order
                                if order is not None and order < len(cr_ordered_splits):
                                    sym_expr, eval_val = cr_ordered_splits[order]
                                    sym_it.extent = sym_expr
                                    for ci_idx, (ev, se) in enumerate(cr_sym_candidates):
                                        if ev == eval_val and se.val == sym_expr.val:
                                            cr_sym_candidates.pop(ci_idx)
                                            break
                                    continue
                            elif stride > 0 and sp_order is not None and rd_order is not None:
                                sp_sym, sp_eval = cr_ordered_splits[sp_order]
                                rd_sym, rd_eval = cr_ordered_splits[rd_order]
                                predicted = (sp_eval - 1) * stride + rd_eval
                                if predicted == real_ext:
                                    stencil_expr = SymExpr(f"({sp_sym.val} - 1)*{stride} + {rd_sym.val}")
                                    sym_it.extent = stencil_expr
                                    continue
                        matched_sym = self._match_cr_extent(real_ext, cr_sym_candidates)
                        if matched_sym is not None:
                            sym_it.extent = matched_sym
                            cr_sym_candidates.remove((real_ext, matched_sym))
                            continue
                    sym_it.extent = SymExpr(real_ext)

    # ─── AnnotationStep ───
    def _apply_annotation(self, step):
        self.s.stages[step.stage_id].iters[step.iter_id].annotation = int(step.annotation)

    # ─── FuseStep ───
    def _apply_fuse(self, step, step_idx):
        s = self.s
        sid = step.stage_id
        fused_ids = [int(x) for x in step.fused_ids]
        stage = s.stages[sid]

        if not fused_ids:
            new_it = SymIter("", SymExpr(1), annotation=0, iter_kind=3)
            stage.iters.insert(0, new_it)
            return

        self._restore_stage_extents_if_needed(sid, step_idx)

        new_name = "@".join(stage.iters[fid].name for fid in fused_ids) + "@"

        new_extent = SymExpr(1)
        all_defined = True
        new_iter_kind = stage.iters[fused_ids[0]].iter_kind
        for fid in fused_ids:
            it = stage.iters[fid]
            if it.extent is not None:
                new_extent = SymExpr.mul(new_extent, it.extent)
            else:
                all_defined = False
            if it.iter_kind != new_iter_kind:
                new_iter_kind = 2

        new_it = SymIter(new_name,
                         new_extent if all_defined else None,
                         annotation=0, iter_kind=new_iter_kind)

        if all_defined and ".shared" in stage.op_name:
            s._shared_fused_extents[sid] = SymExpr(new_extent.val)

        begin = fused_ids[0]
        end = fused_ids[-1]
        new_iters = stage.iters[:begin] + [new_it] + stage.iters[end + 1:]
        stage.iters = new_iters

        removed = len(fused_ids) - 1
        for other_stage in s.stages:
            if (other_stage.compute_at == CA_ITER and
                other_stage.attach_stage_id == sid):
                old_aid = other_stage.attach_iter_id
                if old_aid > end:
                    other_stage.attach_iter_id = old_aid - removed
                elif old_aid >= begin:
                    other_stage.attach_iter_id = begin

    # ─── PragmaStep ───
    def _apply_pragma(self, step, step_idx):
        s = self.s
        sid = step.stage_id
        pragma_type = str(step.pragma_type)
        if pragma_type.startswith("auto_unroll_max_step"):
            parts = pragma_type.split("$")
            if len(parts) == 2:
                val = int(parts[1])
                sym_name = f"ur_{step_idx}"
                s.sym_map[sym_name] = val
                s.stages[sid].auto_unroll_max_step = SymExpr(sym_name)
        elif pragma_type == "debug_skip_region":
            s.stages[sid].compute_at = CA_ROOT
            s.stages[sid].attach_stage_id = None
            s.stages[sid].attach_iter_id = None

    # ─── ReorderStep ───
    def _apply_reorder(self, step):
        s = self.s
        sid = step.stage_id
        after_ids = [int(x) for x in step.after_ids]
        stage = s.stages[sid]
        old_iters = stage.iters
        stage.iters = [old_iters[i] for i in after_ids]
        old_to_new = {old_id: new_id for new_id, old_id in enumerate(after_ids)}
        for other_stage in s.stages:
            if (other_stage.compute_at == CA_ITER and
                other_stage.attach_stage_id == sid and
                other_stage.attach_iter_id in old_to_new):
                other_stage.attach_iter_id = old_to_new[other_stage.attach_iter_id]

    # ─── SplitStep ───
    def _apply_split(self, step, step_idx):
        s = self.s
        sid = step.stage_id
        iid = step.iter_id
        lengths = list(step.lengths)
        inner_to_outer = bool(step.inner_to_outer)

        stage = s.stages[sid]
        self._restore_stage_extents_if_needed(sid, step_idx)

        orig_iter = stage.iters[iid]

        if orig_iter.extent is not None:
            tosplit_extent = orig_iter.extent
        elif step.extent is not None:
            tosplit_extent = SymExpr(int(step.extent))
        else:
            tosplit_extent = None

        sym_lengths = []
        for li, length in enumerate(lengths):
            val = int(length) if length is not None else None
            sym_name = f"sp_{step_idx}_{li}"
            s.sym_map[sym_name] = val
            sym_lengths.append(SymExpr(sym_name))

        sym_prod = SymExpr.product(sym_lengths)
        s._split_sym_products[(sid, step_idx)] = sym_prod

        outs = []
        if inner_to_outer:
            for i in range(len(lengths)):
                li = len(lengths) - i - 1
                name = f"{orig_iter.name}.{len(lengths) - i}"
                sym_ext = sym_lengths[li]
                if tosplit_extent is not None:
                    sym_ext = SymExpr.min(sym_ext, tosplit_extent)
                outs.append(SymIter(name, sym_ext, annotation=0, iter_kind=orig_iter.iter_kind))
                if tosplit_extent is not None:
                    tosplit_extent = SymExpr.ceildiv(tosplit_extent, sym_ext)
                else:
                    tosplit_extent = None
            outs.append(SymIter(f"{orig_iter.name}.0", tosplit_extent,
                                annotation=0, iter_kind=orig_iter.iter_kind))
            outs = list(reversed(outs))
        else:
            for i in range(len(lengths)):
                name = f"{orig_iter.name}.{i}"
                sym_ext = sym_lengths[i]
                if tosplit_extent is not None:
                    sym_ext = SymExpr.min(sym_ext, tosplit_extent)
                outs.append(SymIter(name, sym_ext, annotation=0, iter_kind=orig_iter.iter_kind))
                if tosplit_extent is not None:
                    tosplit_extent = SymExpr.ceildiv(tosplit_extent, sym_ext)
                else:
                    tosplit_extent = None
            outs.append(SymIter(f"{orig_iter.name}.{len(lengths)}", tosplit_extent,
                                annotation=0, iter_kind=orig_iter.iter_kind))

        new_iters = stage.iters[:iid] + outs + stage.iters[iid + 1:]
        stage.iters = new_iters

        shift = len(lengths)
        for other_stage in s.stages:
            if (other_stage.compute_at == CA_ITER and
                other_stage.attach_stage_id == sid and
                other_stage.attach_iter_id >= iid):
                other_stage.attach_iter_id += shift

    # ─── FollowSplitStep ───
    def _apply_follow_split(self, step, all_steps, step_idx):
        s = self.s
        sid = step.stage_id
        iid = step.iter_id
        src_step_id = step.src_step_id
        n_split = int(step.n_split)

        src_step = all_steps[src_step_id]
        src_lengths = list(src_step.lengths)

        extracted = []
        for j in range(min(n_split - 1, len(src_lengths))):
            extracted.append(src_lengths[j])
        last = 1
        j_start = n_split - 1
        all_defined = True
        for j in range(j_start, len(src_lengths)):
            if src_lengths[j] is not None:
                last *= int(src_lengths[j])
            else:
                all_defined = False
                break
        extracted.append(last if all_defined else None)

        stage = s.stages[sid]
        self._restore_stage_extents_if_needed(sid, step_idx)

        orig_iter = stage.iters[iid]
        orig_extent = None
        if orig_iter.extent is not None and orig_iter.extent.is_concrete:
            orig_extent = orig_iter.extent.val

        tosplit_extent = SymExpr(orig_extent) if orig_extent is not None else \
                         (orig_iter.extent if orig_iter.extent is not None else None)
        sym_lengths = []
        for li in range(len(extracted)):
            if li < n_split - 1:
                src_sym_name = f"sp_{src_step_id}_{li}"
            else:
                if j_start < len(src_lengths) and j_start == len(src_lengths) - 1:
                    src_sym_name = f"sp_{src_step_id}_{j_start}"
                else:
                    parts = [f"sp_{src_step_id}_{j}" for j in range(j_start, len(src_lengths))]
                    src_sym_name = "*".join(parts) if parts else str(extracted[li])
            sym_lengths.append(SymExpr(src_sym_name))

        outs = []
        for i in range(len(extracted)):
            li = len(extracted) - i - 1
            name = f"{orig_iter.name}.{len(extracted) - i}"
            sym_ext = sym_lengths[li]
            if tosplit_extent is not None:
                sym_ext = SymExpr.min(sym_ext, tosplit_extent)
            outs.append(SymIter(name, sym_ext, annotation=0, iter_kind=orig_iter.iter_kind))
            if tosplit_extent is not None:
                tosplit_extent = SymExpr.ceildiv(tosplit_extent, sym_ext)
            else:
                tosplit_extent = None
        outs.append(SymIter(f"{orig_iter.name}.0", tosplit_extent,
                            annotation=0, iter_kind=orig_iter.iter_kind))
        outs = list(reversed(outs))

        new_iters = stage.iters[:iid] + outs + stage.iters[iid + 1:]
        stage.iters = new_iters

        shift = len(extracted)
        for other_stage in s.stages:
            if (other_stage.compute_at == CA_ITER and
                other_stage.attach_stage_id == sid and
                other_stage.attach_iter_id >= iid):
                other_stage.attach_iter_id += shift

    # ─── FollowFusedSplitStep ───
    def _apply_follow_fused_split(self, step, all_steps, step_idx):
        s = self.s
        sid = step.stage_id
        iid = step.iter_id
        src_step_ids = [int(x) for x in step.src_step_ids]
        level = int(step.level)
        factor_or_nparts = bool(step.factor_or_nparts)

        stage = s.stages[sid]
        self._restore_stage_extents_if_needed(sid, step_idx)

        orig_iter = stage.iters[iid]
        orig_extent = None
        if orig_iter.extent is not None and orig_iter.extent.is_concrete:
            orig_extent = orig_iter.extent.val

        tosplit_extent = SymExpr(orig_extent) if orig_extent is not None else \
                         (orig_iter.extent if orig_iter.extent is not None else None)

        src_sym_parts = []
        for sid_ref in src_step_ids:
            src_sym_parts.append(f"sp_{sid_ref}_{level}")
        fused_sym_expr = SymExpr("*".join(src_sym_parts) if len(src_sym_parts) > 1 else src_sym_parts[0])

        if factor_or_nparts:
            inner_ext = fused_sym_expr
            outer_ext = SymExpr.ceildiv(tosplit_extent, fused_sym_expr) if tosplit_extent else None
            outs = [
                SymIter(f"{orig_iter.name}.0", outer_ext, annotation=0, iter_kind=orig_iter.iter_kind),
                SymIter(f"{orig_iter.name}.1", inner_ext, annotation=0, iter_kind=orig_iter.iter_kind),
            ]
        else:
            outer_ext = fused_sym_expr
            inner_ext = SymExpr.ceildiv(tosplit_extent, fused_sym_expr) if tosplit_extent else None
            outs = [
                SymIter(f"{orig_iter.name}.0", outer_ext, annotation=0, iter_kind=orig_iter.iter_kind),
                SymIter(f"{orig_iter.name}.1", inner_ext, annotation=0, iter_kind=orig_iter.iter_kind),
            ]

        new_iters = stage.iters[:iid] + outs + stage.iters[iid + 1:]
        stage.iters = new_iters

        for other_stage in s.stages:
            if (other_stage.compute_at == CA_ITER and
                other_stage.attach_stage_id == sid and
                other_stage.attach_iter_id >= iid):
                other_stage.attach_iter_id += 1

    # ─── StorageAlignStep ───
    def _apply_storage_align(self, step):
        self.s.stages[step.stage_id].storage_offset = step.offset

    # ─── ComputeAtStep ───
    def _apply_compute_at(self, step):
        s = self.s
        sid = step.stage_id
        target_sid = step.target_stage_id
        target_iid = step.target_iter_id
        stage = s.stages[sid]

        for iid, it in enumerate(stage.iters):
            if it.extent is not None and not it.extent.is_concrete:
                s._ca_saved_extents[(sid, iid)] = SymExpr(it.extent.val)
            it.extent = None

        stage.compute_at = CA_ITER
        stage.attach_stage_id = target_sid
        stage.attach_iter_id = target_iid

    # ─── ComputeInlineStep ───
    def _apply_compute_inline(self, step):
        stage = self.s.stages[step.stage_id]
        stage.compute_at = CA_INLINED
        stage.attach_stage_id = None
        stage.attach_iter_id = None

    # ─── ComputeRootStep ───
    def _apply_compute_root(self, step):
        s = self.s
        sid = step.stage_id
        stage = s.stages[sid]
        for iid, it in enumerate(stage.iters):
            if it.extent is not None and not it.extent.is_concrete:
                s._ca_saved_extents[(sid, iid)] = SymExpr(it.extent.val)
            it.extent = None
        stage.compute_at = CA_ROOT
        stage.attach_stage_id = None
        stage.attach_iter_id = None

    # ─── CacheReadStep ───
    def _apply_cache_read(self, step, state, step_idx):
        s = self.s
        sid = step.stage_id
        added_stage_id = sid + 1

        ps_after = tvm._ffi.get_global_func("auto_scheduler.ReplayStepsPartial")(
            s.compute_dag, state, step_idx + 1)

        new_stage_real = ps_after.stages[added_stage_id]
        new_iters = []
        for it in new_stage_real.iters:
            ext = int(it.range.extent) if it.range is not None else None
            new_iters.append(SymIter(it.name, SymExpr(ext) if ext is not None else None,
                                      annotation=0, iter_kind=0))

        new_sym_stage = SymStage(new_stage_real.op.name, 'compute', new_iters,
                                 dtype=str(new_stage_real.op.output(0).dtype)
                                       if hasattr(new_stage_real.op, 'output') else "float32")
        s.stages.insert(added_stage_id, new_sym_stage)

        s._shift_ca_saved_extents(added_stage_id)

        reader_ids = [int(x) for x in step.reader_stage_ids]
        if reader_ids:
            consumer_sid = reader_ids[0]
            if consumer_sid >= added_stage_id:
                consumer_sid += 1
            s._cache_read_consumer[added_stage_id] = consumer_sid
            self._analyze_cache_read_stencil(added_stage_id, sid, consumer_sid)

        for other_stage in s.stages:
            if other_stage.compute_at == CA_ITER:
                if other_stage.attach_stage_id is not None and other_stage.attach_stage_id >= added_stage_id:
                    other_stage.attach_stage_id += 1

        for i in range(added_stage_id + 1, len(s.stages)):
            real_stage = ps_after.stages[i]
            s.stages[i].op_name = real_stage.op.name

    def _analyze_cache_read_stencil(self, cr_stage_id, orig_tensor_sid, consumer_sid):
        s = self.s
        from tvm import tir

        cr_name = s.stages[cr_stage_id].op_name
        orig_name = cr_name.rsplit(".", 1)[0] if "." in cr_name else cr_name

        consumer_name = s.stages[consumer_sid].op_name
        consumer_orig_name = consumer_name.rsplit(".", 1)[0] if "." in consumer_name else consumer_name

        consumer_op = None
        for op in s.compute_dag.ops:
            if op.name == consumer_orig_name or op.name == consumer_name:
                if hasattr(op, 'body'):
                    consumer_op = op
                    break

        if consumer_op is None or not hasattr(consumer_op, 'body') or len(consumer_op.body) == 0:
            return

        producer_loads = self._find_producer_loads(consumer_op.body[0], orig_name)
        if not producer_loads:
            return

        pl = producer_loads[0]

        spatial_vars = {str(ax.var.name): i for i, ax in enumerate(consumer_op.axis)}
        reduce_vars = {str(ax.var.name): len(consumer_op.axis) + i
                      for i, ax in enumerate(consumer_op.reduce_axis)}

        stencil_info = {}
        for ax_idx, idx_expr in enumerate(pl.indices):
            result = self._analyze_index_expr(idx_expr, consumer_op.axis, consumer_op.reduce_axis)
            if result is None:
                continue
            stride, sp_name, rd_name = result
            sp_order = spatial_vars.get(sp_name) if sp_name else None
            rd_order = reduce_vars.get(rd_name) if rd_name else None
            stencil_info[ax_idx] = (stride, sp_order, rd_order)

        if stencil_info:
            s._cache_read_stencil_info[cr_stage_id] = stencil_info

    @staticmethod
    def _find_producer_loads(expr, tensor_name):
        from tvm import tir
        results = []
        def visit(e):
            if isinstance(e, tir.ProducerLoad):
                if str(e.producer.name) == tensor_name:
                    results.append(e)
                return
            if isinstance(e, tir.Reduce):
                for src in e.source:
                    visit(src)
                return
            for attr in ['a', 'b']:
                if hasattr(e, attr):
                    visit(getattr(e, attr))
            if hasattr(e, 'args') and not isinstance(e, (tir.Var, tir.IntImm)):
                for arg in e.args:
                    visit(arg)
        visit(expr)
        return results

    @staticmethod
    def _analyze_index_expr(expr, spatial_axes, reduce_axes):
        from tvm import tir
        spatial_names = {str(v.var.name) for v in spatial_axes}
        reduce_names = {str(v.var.name) for v in reduce_axes}

        if isinstance(expr, tir.Var):
            name = str(expr.name)
            if name in spatial_names:
                return (0, name, None)
            elif name in reduce_names:
                return (0, None, name)
            return None

        if isinstance(expr, tir.Add):
            a, b = expr.a, expr.b
            def extract_mul_var(e):
                if isinstance(e, tir.Mul):
                    if isinstance(e.a, tir.Var) and isinstance(e.b, tir.IntImm):
                        return (str(e.a.name), int(e.b))
                    if isinstance(e.b, tir.Var) and isinstance(e.a, tir.IntImm):
                        return (str(e.b.name), int(e.a))
                elif isinstance(e, tir.Var):
                    return (str(e.name), 1)
                return None

            mul_info = extract_mul_var(a)
            if mul_info and isinstance(b, tir.Var):
                sp_name, stride = mul_info
                rd_name = str(b.name)
                if sp_name in spatial_names and rd_name in reduce_names:
                    return (stride, sp_name, rd_name)

            mul_info = extract_mul_var(b)
            if mul_info and isinstance(a, tir.Var):
                sp_name, stride = mul_info
                rd_name = str(a.name)
                if sp_name in spatial_names and rd_name in reduce_names:
                    return (stride, sp_name, rd_name)

            if isinstance(a, tir.Var) and isinstance(b, tir.Var):
                a_name, b_name = str(a.name), str(b.name)
                if a_name in spatial_names and b_name in reduce_names:
                    return (1, a_name, b_name)
                if b_name in spatial_names and a_name in reduce_names:
                    return (1, b_name, a_name)

        return None

    # ─── CacheWriteStep ───
    def _apply_cache_write(self, step, state, step_idx):
        s = self.s
        sid = step.stage_id
        ps_after = tvm._ffi.get_global_func("auto_scheduler.ReplayStepsPartial")(
            s.compute_dag, state, step_idx + 1)

        new_stage_real = ps_after.stages[sid]
        new_iters = []
        for it in new_stage_real.iters:
            ext = int(it.range.extent) if it.range is not None else None
            new_iters.append(SymIter(it.name, SymExpr(ext) if ext is not None else None,
                                      annotation=0, iter_kind=0))

        new_sym_stage = SymStage(new_stage_real.op.name, 'compute', new_iters,
                                 dtype=str(new_stage_real.op.output(0).dtype)
                                       if hasattr(new_stage_real.op, 'output') else "float32")
        s.stages.insert(sid, new_sym_stage)

        s._shift_ca_saved_extents(sid)

        for other_stage in s.stages:
            if other_stage.compute_at == CA_ITER:
                if other_stage.attach_stage_id is not None and other_stage.attach_stage_id >= sid:
                    other_stage.attach_stage_id += 1

        for i in range(len(s.stages)):
            if i < len(ps_after.stages):
                real_stage = ps_after.stages[i]
                s.stages[i].op_name = real_stage.op.name


# ─────────────────────────────────────────────────────────────
#  SymParamManager: 파라미터 조회 / 검증 / 랜덤 생성
# ─────────────────────────────────────────────────────────────
class SymParamManager:
    """
    SymbolicState의 sym_map 파라미터를 조회·검증·수정하는 매니저.
    """

    UNROLL_CANDIDATES = [0, 16, 64, 512, 1024]

    def __init__(self, sym_state):
        """
        Args:
            sym_state: SymbolicState 인스턴스
        """
        self.s = sym_state

    def _build_sp_groups(self):
        """SP 그룹 구성: step_idx → [sym_name, ...] (length_idx 순 정렬)."""
        sp_groups = {}
        for name in self.s.sym_map:
            if name.startswith("sp_"):
                parts = name.split("_")
                step_idx = int(parts[1])
                sp_groups.setdefault(step_idx, []).append(name)
        for step_idx in sp_groups:
            sp_groups[step_idx].sort(key=lambda n: int(n.split("_")[2]))
        return sp_groups

    def _build_sp_extents(self, sp_groups):
        """SP step_idx → extent 매핑 (SplitStep의 원본 extent)."""
        sp_extents = {}
        if self.s._state is not None:
            steps = self.s._state.transform_steps
            for step_idx in sp_groups:
                if step_idx < len(steps):
                    step = steps[step_idx]
                    tk = step.type_key.split(".")[-1]
                    if tk == "SplitStep" and step.extent is not None:
                        sp_extents[step_idx] = int(step.extent)
        return sp_extents

    def get_sym_params(self):
        """sym_map의 모든 symbolic parameter 정보를 반환.

        Returns: list of dict, 각 항목:
          - name, kind("sp"|"ur"), value, step_idx, length_idx,
            sp_group, extent, candidates
        """
        sp_groups = self._build_sp_groups()
        sp_extents = self._build_sp_extents(sp_groups)

        results = []
        for name, value in self.s.sym_map.items():
            if name.startswith("sp_"):
                parts = name.split("_")
                step_idx = int(parts[1])
                length_idx = int(parts[2])
                group = sp_groups.get(step_idx, [name])
                extent = sp_extents.get(step_idx)
                candidates = self._get_sp_candidates(step_idx, length_idx, sp_groups, extent)
                results.append({
                    "name": name, "kind": "sp", "value": value,
                    "step_idx": step_idx, "length_idx": length_idx,
                    "sp_group": group, "extent": extent, "candidates": candidates,
                })
            elif name.startswith("ur_"):
                step_idx = int(name.split("_")[1])
                results.append({
                    "name": name, "kind": "ur", "value": value,
                    "step_idx": step_idx, "length_idx": None,
                    "sp_group": None, "extent": None,
                    "candidates": list(self.UNROLL_CANDIDATES),
                })
        return results

    def _get_sp_candidates(self, step_idx, length_idx, sp_groups, extent):
        """SP 파라미터의 약수 후보 계산 (좌→우 순서)."""
        if extent is None:
            return []
        group = sp_groups.get(step_idx, [])
        remaining = extent
        for name in group:
            li = int(name.split("_")[2])
            if li < length_idx:
                val = self.s.sym_map.get(name)
                if val is not None and val > 0:
                    remaining = (remaining + val - 1) // val
                else:
                    return []
            elif li == length_idx:
                break
        return self._divisors(remaining)

    @staticmethod
    def _divisors(n):
        """양의 정수 n의 약수 목록을 오름차순으로 반환."""
        if n <= 0:
            return [1]
        divs = []
        for i in range(1, int(n**0.5) + 1):
            if n % i == 0:
                divs.append(i)
                if i != n // i:
                    divs.append(n // i)
        return sorted(divs)

    def set_sym_params(self, param_dict):
        """파라미터 값 일괄 설정 (약수 조건 검증 포함).
        Raises: ValueError on constraint violation.
        """
        sp_groups = self._build_sp_groups()
        sp_extents = self._build_sp_extents(sp_groups)

        backup = dict(self.s.sym_map)
        try:
            for name, value in param_dict.items():
                if name not in self.s.sym_map:
                    raise ValueError(f"Unknown param '{name}' (not in sym_map)")
                if name.startswith("ur_"):
                    if value not in self.UNROLL_CANDIDATES:
                        raise ValueError(
                            f"Unroll param '{name}' value {value} not in {self.UNROLL_CANDIDATES}")
                    self.s.sym_map[name] = value
                elif name.startswith("sp_"):
                    parts = name.split("_")
                    step_idx = int(parts[1])
                    length_idx = int(parts[2])
                    extent = sp_extents.get(step_idx)
                    if extent is not None:
                        candidates = self._get_sp_candidates(
                            step_idx, length_idx, sp_groups, extent)
                        if value not in candidates:
                            raise ValueError(
                                f"SP param '{name}' value {value} not a valid divisor. "
                                f"candidates={candidates}")
                    self.s.sym_map[name] = value
                else:
                    self.s.sym_map[name] = value
        except ValueError:
            self.s.sym_map.update(backup)
            raise

    def get_sym_params_summary(self):
        """파라미터 요약을 읽기 좋은 문자열로 반환."""
        params = self.get_sym_params()
        lines = []
        sp_by_step = {}
        for p in params:
            if p["kind"] == "sp":
                sp_by_step.setdefault(p["step_idx"], []).append(p)
        for step_idx in sorted(sp_by_step.keys()):
            group = sp_by_step[step_idx]
            ext = group[0]["extent"]
            vals = [str(p["value"]) for p in group]
            names = [p["name"] for p in group]
            lines.append(f"  SP step={step_idx} extent={ext}: "
                        f"{', '.join(names)} = [{', '.join(vals)}]")
        for p in params:
            if p["kind"] == "ur":
                lines.append(f"  UR step={p['step_idx']}: {p['name']} = {p['value']}")
        return "\n".join(lines)

    def randomize_params(self, rng=None):
        """모든 파라미터를 약수 조건을 만족하도록 랜덤으로 채운다.

        Args:
            rng: random.Random 인스턴스 또는 None

        Returns:
            dict: 설정된 {sym_name: value} 매핑
        """
        import random as _random
        if rng is None:
            rng = _random.Random()

        sp_groups = self._build_sp_groups()
        sp_extents = self._build_sp_extents(sp_groups)
        ur_names = [n for n in self.s.sym_map if n.startswith("ur_")]

        result = {}

        for step_idx in sorted(sp_groups.keys()):
            group = sp_groups[step_idx]
            extent = sp_extents.get(step_idx)
            if extent is None:
                for name in group:
                    result[name] = self.s.sym_map[name]
                continue

            remaining = extent
            for name in group:
                candidates = self._divisors(remaining)
                chosen = rng.choice(candidates)
                self.s.sym_map[name] = chosen
                result[name] = chosen
                remaining = (remaining + chosen - 1) // chosen

        for name in ur_names:
            chosen = rng.choice(self.UNROLL_CANDIDATES)
            self.s.sym_map[name] = chosen
            result[name] = chosen

        return result


# ─────────────────────────────────────────────────────────────
#  ExprNode: 제약식 표현식 트리 (interval arithmetic 지원)
# ─────────────────────────────────────────────────────────────
class ExprNode:
    """제약식 LHS를 나타내는 표현식 트리 노드.

    각 노드는 partial assignment 상태에서 (lo, hi) 구간을 반환할 수 있다.
    lo = 현재 도메인 하에서 최소값, hi = 현재 도메인 하에서 최대값.
    """
    pass

class ConstNode(ExprNode):
    """정수 상수."""
    __slots__ = ('val',)
    def __init__(self, val):
        self.val = int(val)

    def interval(self, domains):
        return (self.val, self.val)

    def evaluate(self, assignment):
        return self.val

    def variables(self):
        return set()

    def __repr__(self):
        return str(self.val)

class VarNode(ExprNode):
    """심볼릭 변수 (sp_X_Y)."""
    __slots__ = ('name',)
    def __init__(self, name):
        self.name = name

    def interval(self, domains):
        dom = domains.get(self.name)
        if dom is None:
            return (1, 1)
        if isinstance(dom, int):
            return (dom, dom)
        # dom은 정렬된 리스트
        return (dom[0], dom[-1])

    def evaluate(self, assignment):
        return assignment.get(self.name, 1)

    def variables(self):
        return {self.name}

    def __repr__(self):
        return self.name

class MulNode(ExprNode):
    """곱셈: left * right."""
    __slots__ = ('left', 'right')
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def interval(self, domains):
        a_lo, a_hi = self.left.interval(domains)
        b_lo, b_hi = self.right.interval(domains)
        # 모든 인수는 ≥1 이므로 단조증가
        return (a_lo * b_lo, a_hi * b_hi)

    def evaluate(self, assignment):
        return self.left.evaluate(assignment) * self.right.evaluate(assignment)

    def variables(self):
        return self.left.variables() | self.right.variables()

    def __repr__(self):
        return f"({self.left}*{self.right})"

class AddNode(ExprNode):
    """덧셈: left + right."""
    __slots__ = ('left', 'right')
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def interval(self, domains):
        a_lo, a_hi = self.left.interval(domains)
        b_lo, b_hi = self.right.interval(domains)
        return (a_lo + b_lo, a_hi + b_hi)

    def evaluate(self, assignment):
        return self.left.evaluate(assignment) + self.right.evaluate(assignment)

    def variables(self):
        return self.left.variables() | self.right.variables()

    def __repr__(self):
        return f"({self.left}+{self.right})"

class SubNode(ExprNode):
    """뺄셈: left - right."""
    __slots__ = ('left', 'right')
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def interval(self, domains):
        a_lo, a_hi = self.left.interval(domains)
        b_lo, b_hi = self.right.interval(domains)
        return (a_lo - b_hi, a_hi - b_lo)

    def evaluate(self, assignment):
        return self.left.evaluate(assignment) - self.right.evaluate(assignment)

    def variables(self):
        return self.left.variables() | self.right.variables()

    def __repr__(self):
        return f"({self.left}-{self.right})"

class MinNode(ExprNode):
    """min(left, right)."""
    __slots__ = ('left', 'right')
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def interval(self, domains):
        a_lo, a_hi = self.left.interval(domains)
        b_lo, b_hi = self.right.interval(domains)
        return (builtins_min(a_lo, b_lo), builtins_min(a_hi, b_hi))

    def evaluate(self, assignment):
        return builtins_min(self.left.evaluate(assignment), self.right.evaluate(assignment))

    def variables(self):
        return self.left.variables() | self.right.variables()

    def __repr__(self):
        return f"min({self.left},{self.right})"

class CeilDivNode(ExprNode):
    """ceil(left / right)."""
    __slots__ = ('left', 'right')
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def interval(self, domains):
        a_lo, a_hi = self.left.interval(domains)
        b_lo, b_hi = self.right.interval(domains)
        # ceil(a/b): lo when a small and b large, hi when a large and b small
        b_lo = max(b_lo, 1)  # 0 나누기 방지
        b_hi = max(b_hi, 1)
        lo = (a_lo + b_hi - 1) // b_hi
        hi = (a_hi + b_lo - 1) // b_lo
        return (max(lo, 1), max(hi, 1))

    def evaluate(self, assignment):
        a = self.left.evaluate(assignment)
        b = self.right.evaluate(assignment)
        b = max(b, 1)
        return (a + b - 1) // b

    def variables(self):
        return self.left.variables() | self.right.variables()

    def __repr__(self):
        return f"ceil({self.left}/{self.right})"

class ScaleMulNode(ExprNode):
    """child * constant (dtype_bytes 등 곱). 최적화된 단축형."""
    __slots__ = ('child', 'scale')
    def __init__(self, child, scale):
        self.child = child
        self.scale = scale

    def interval(self, domains):
        lo, hi = self.child.interval(domains)
        return (lo * self.scale, hi * self.scale)

    def evaluate(self, assignment):
        return self.child.evaluate(assignment) * self.scale

    def variables(self):
        return self.child.variables()

    def __repr__(self):
        return f"({self.child}*{self.scale})"

class SumNode(ExprNode):
    """여러 자식의 합. shared memory 총합에 사용."""
    __slots__ = ('children',)
    def __init__(self, children):
        self.children = list(children)

    def interval(self, domains):
        lo = sum(c.interval(domains)[0] for c in self.children)
        hi = sum(c.interval(domains)[1] for c in self.children)
        return (lo, hi)

    def evaluate(self, assignment):
        return sum(c.evaluate(assignment) for c in self.children)

    def variables(self):
        v = set()
        for c in self.children:
            v |= c.variables()
        return v

    def __repr__(self):
        return " + ".join(str(c) for c in self.children)


def _parse_expr_tree(sym_expr_str):
    """SymExpr 문자열을 ExprNode 트리로 파싱.

    지원하는 문법:
      - 정수 리터럴
      - sp_X_Y 변수
      - a*b (곱)
      - (expr)
      - min(a,b)
      - ceil(a/(b))
      - a - b, a + b (스텐실 패턴)

    Returns: ExprNode
    """
    s = sym_expr_str.strip()
    return _parse_add_sub(s, 0)[0]

def _parse_add_sub(s, pos):
    """+ / - 파싱 (가장 낮은 우선순위)."""
    left, pos = _parse_mul(s, pos)
    while pos < len(s):
        if s[pos] == '+':
            right, pos = _parse_mul(s, pos + 1)
            left = AddNode(left, right)
        elif s[pos] == '-' and pos > 0:
            right, pos = _parse_mul(s, pos + 1)
            left = SubNode(left, right)
        elif s[pos] in ' ':
            # " - " 또는 " + " 패턴 처리
            j = pos
            while j < len(s) and s[j] == ' ':
                j += 1
            if j < len(s) and s[j] == '+':
                j2 = j + 1
                while j2 < len(s) and s[j2] == ' ':
                    j2 += 1
                right, pos = _parse_mul(s, j2)
                left = AddNode(left, right)
            elif j < len(s) and s[j] == '-':
                j2 = j + 1
                while j2 < len(s) and s[j2] == ' ':
                    j2 += 1
                right, pos = _parse_mul(s, j2)
                left = SubNode(left, right)
            else:
                break
        else:
            break
    return left, pos

def _parse_mul(s, pos):
    """* 파싱."""
    left, pos = _parse_atom(s, pos)
    while pos < len(s) and s[pos] == '*':
        right, pos = _parse_atom(s, pos + 1)
        left = MulNode(left, right)
    return left, pos

def _parse_atom(s, pos):
    """원자: 정수, 변수, 괄호, min(...), ceil(...)."""
    # 공백 스킵
    while pos < len(s) and s[pos] == ' ':
        pos += 1

    if pos >= len(s):
        return ConstNode(1), pos

    # 괄호
    if s[pos] == '(':
        inner, pos = _parse_add_sub(s, pos + 1)
        if pos < len(s) and s[pos] == ')':
            pos += 1
        return inner, pos

    # min(...)
    if s[pos:pos+4] == 'min(':
        pos += 4
        a, pos = _parse_add_sub(s, pos)
        if pos < len(s) and s[pos] == ',':
            pos += 1
        b, pos = _parse_add_sub(s, pos)
        if pos < len(s) and s[pos] == ')':
            pos += 1
        return MinNode(a, b), pos

    # ceil(.../(...)...)
    if s[pos:pos+5] == 'ceil(':
        pos += 5
        a, pos = _parse_add_sub(s, pos)
        if pos < len(s) and s[pos] == '/':
            pos += 1
        # slash 뒤에 ( 가 올 수 있음
        if pos < len(s) and s[pos] == '(':
            b, pos = _parse_add_sub(s, pos + 1)
            if pos < len(s) and s[pos] == ')':
                pos += 1
        else:
            b, pos = _parse_atom(s, pos)
        if pos < len(s) and s[pos] == ')':
            pos += 1
        return CeilDivNode(a, b), pos

    # math.ceil(...) — eval_sym_extent가 생성하는 형태
    if s[pos:pos+10] == 'math.ceil(':
        pos += 10
        a, pos = _parse_add_sub(s, pos)
        if pos < len(s) and s[pos] == '/':
            pos += 1
        if pos < len(s) and s[pos] == '(':
            b, pos = _parse_add_sub(s, pos + 1)
            if pos < len(s) and s[pos] == ')':
                pos += 1
        else:
            b, pos = _parse_atom(s, pos)
        if pos < len(s) and s[pos] == ')':
            pos += 1
        return CeilDivNode(a, b), pos

    # 정수 또는 변수
    start = pos
    if s[pos].isdigit():
        while pos < len(s) and s[pos].isdigit():
            pos += 1
        return ConstNode(int(s[start:pos])), pos

    # 변수: sp_X_Y 등 (알파벳/밑줄/숫자)
    if s[pos].isalpha() or s[pos] == '_':
        while pos < len(s) and (s[pos].isalnum() or s[pos] == '_'):
            pos += 1
        name = s[start:pos]
        return VarNode(name), pos

    # 알 수 없는 문자 — 스킵
    return ConstNode(1), pos + 1


# ─────────────────────────────────────────────────────────────
#  ScheduleGenerator: HW 제약 기반 스케줄 파라미터 생성
# ─────────────────────────────────────────────────────────────
class ScheduleGenerator:
    """
    HW_PARAM 기반 제약식을 구축하고, 제약을 만족하는 파라미터를 생성하는 생성기.

    제약 목록:
      1) max_vectorize_bytes:  각 vectorize extent × dtype_bytes ≤ max_vector_bytes
      2) max_shared_memory:    block 당 shared memory 총합 ≤ max_shared_memory_per_block
      3) max_threads:          각 thread extent ≤ max_threads_per_block
      4) min_thread_extent:    spatial product > warp_size*2 이면 thread extent ≥ warp_size
      5) max_vthread:          각 vthread extent ≤ max_vthread_extent
      6) max_innermost_split:  각 SplitStep의 마지막 factor ≤ max_innermost_split_factor

    생성 알고리즘:
      1) 초기화: 각 변수의 약수 도메인 구성 + max_innermost_split 반영
      2) 제약 전처리: 표현식 트리 파싱 + interval arithmetic
      3) 변수 순서 결정: shared → thread → 나머지
      4) 순차 할당: LHS_min > RHS 이면 후보 제거
      5) 상태 업데이트: 관련 제약식만 갱신
    """

    DEFAULT_HW_PARAM = {
        'max_vector_bytes': 16,
        'max_shared_memory_per_block': 49152,
        'max_threads_per_block': 1024,
        'max_vthread_extent': 8,
        'warp_size': 32,
        'max_innermost_split_factor': 64,
    }

    def __init__(self, sym_state, hw_param=None):
        """
        Args:
            sym_state: SymbolicState 인스턴스 (apply_steps 완료 상태)
            hw_param: dict — HW 제약 파라미터. None이면 DEFAULT_HW_PARAM 사용.
        """
        self.s = sym_state
        self.hw = dict(self.DEFAULT_HW_PARAM)
        if hw_param is not None:
            self.hw.update(hw_param)
        self.pm = SymParamManager(sym_state)

        # ─── 전처리: 제약식 빌드 + 파싱 ───
        self._constraints = []   # list of parsed constraint dicts
        self._var_constraints = {}  # {var_name: [constraint_index, ...]}
        self._initial_domains = {}  # {var_name: sorted list of candidates}
        self._var_order = []     # 변수 할당 순서

        self._preprocess()

    # ═══════════════════════════════════════════════════════════
    # 1) 제약식 빌드 (기존 API 유지)
    # ═══════════════════════════════════════════════════════════

    # ─── 1) Vectorize constraint ───
    def build_vectorize_constraints(self):
        """각 vectorize iter에 대해 (sym_extent_expr, dtype_bytes, limit) 튜플 목록 반환.
        제약: eval(sym_extent) * dtype_bytes ≤ max_vector_bytes
        """
        limit = self.hw['max_vector_bytes']
        constraints = []
        for sid, iid, ext in self.s.get_vectorize_extents():
            dtype_bytes = self.s.stages[sid].dtype_bytes
            constraints.append({
                'stage_id': sid,
                'iter_id': iid,
                'sym_extent': ext,
                'dtype_bytes': dtype_bytes,
                'limit': limit,
                'desc': f"vectorize s{sid}.i{iid} ({self.s.stages[sid].op_name}): "
                        f"extent*{dtype_bytes} ≤ {limit}",
            })
        return constraints

    def check_vectorize(self, sym_map=None):
        """vectorize 제약 위반 목록 반환. 빈 리스트 = 통과."""
        if sym_map is None:
            sym_map = self.s.sym_map
        violations = []
        for c in self.build_vectorize_constraints():
            val = eval_sym_extent(c['sym_extent'], sym_map)
            if isinstance(val, int) and val * c['dtype_bytes'] > c['limit']:
                violations.append(f"{c['desc']}: actual={val}*{c['dtype_bytes']}={val*c['dtype_bytes']}")
        return violations

    # ─── 2) Shared memory constraint ───
    def build_shared_memory_constraints(self):
        """block 당 shared memory 총합 제약.
        각 .shared stage의 extent × dtype_bytes 의 합 ≤ limit.
        Returns: dict with sym_exprs list, dtype_bytes list, limit.
        """
        limit = self.hw['max_shared_memory_per_block']
        items = []
        for sid, op_name, ext in self.s.get_shared_memory_extents():
            dtype_bytes = self.s.stages[sid].dtype_bytes
            items.append({
                'stage_id': sid,
                'op_name': op_name,
                'sym_extent': ext,
                'dtype_bytes': dtype_bytes,
            })
        return {
            'items': items,
            'limit': limit,
            'desc': f"shared_memory: sum(extent*dtype_bytes) ≤ {limit}",
        }

    def check_shared_memory(self, sym_map=None):
        """shared memory 제약 위반 여부. 위반 시 문자열 리스트 반환."""
        if sym_map is None:
            sym_map = self.s.sym_map
        c = self.build_shared_memory_constraints()
        total = 0
        parts = []
        for item in c['items']:
            val = eval_sym_extent(item['sym_extent'], sym_map)
            if isinstance(val, int):
                bytes_used = val * item['dtype_bytes']
                total += bytes_used
                parts.append(f"{item['op_name']}={val}*{item['dtype_bytes']}={bytes_used}")
        if total > c['limit']:
            return [f"{c['desc']}: actual={total} ({' + '.join(parts)})"]
        return []

    # ─── 3) Max threads per block constraint ───
    def build_max_threads_constraints(self):
        """각 thread extent ≤ max_threads_per_block 제약 목록."""
        limit = self.hw['max_threads_per_block']
        constraints = []
        for sid, iid, ext in self.s.get_thread_extents():
            constraints.append({
                'stage_id': sid,
                'iter_id': iid,
                'sym_extent': ext,
                'limit': limit,
                'desc': f"max_threads s{sid}.i{iid} ({self.s.stages[sid].op_name}): "
                        f"extent ≤ {limit}",
            })
        return constraints

    def check_max_threads(self, sym_map=None):
        """max_threads 제약 위반 목록."""
        if sym_map is None:
            sym_map = self.s.sym_map
        violations = []
        for c in self.build_max_threads_constraints():
            val = eval_sym_extent(c['sym_extent'], sym_map)
            if isinstance(val, int) and val > c['limit']:
                violations.append(f"{c['desc']}: actual={val}")
        return violations

    # ─── 4) Min thread extent constraint ───
    def build_min_thread_constraints(self):
        """spatial product > warp_size*2 인 stage의 thread extent ≥ warp_size 제약 목록.
        C++ InitThreadBind의 check_min_thread_extent 로직과 동일."""
        warp_size = self.hw['warp_size']
        constraints = []
        for sid, iid, ext in self.s.get_thread_extents():
            # _state.stages에서 원본 op의 spatial axis product 계산
            if self.s._state is not None and sid < len(self.s._state.stages):
                real_stg = self.s._state.stages[sid]
                op = real_stg.op
                if hasattr(op, 'axis') and len(op.axis) > 0:
                    total_space = 1
                    for ax in op.axis:
                        if ax.dom is not None:
                            total_space *= int(ax.dom.extent)
                    if total_space > warp_size * 2:
                        constraints.append({
                            'stage_id': sid,
                            'iter_id': iid,
                            'sym_extent': ext,
                            'min_val': warp_size,
                            'total_space': total_space,
                            'desc': f"min_thread s{sid}.i{iid} ({self.s.stages[sid].op_name}): "
                                    f"extent ≥ {warp_size} (space={total_space})",
                        })
        return constraints

    def check_min_thread(self, sym_map=None):
        """min_thread 제약 위반 목록."""
        if sym_map is None:
            sym_map = self.s.sym_map
        violations = []
        for c in self.build_min_thread_constraints():
            val = eval_sym_extent(c['sym_extent'], sym_map)
            if isinstance(val, int) and val < c['min_val']:
                violations.append(f"{c['desc']}: actual={val}")
        return violations

    # ─── 5) Max vthread extent constraint ───
    def build_vthread_constraints(self):
        """각 vthread extent ≤ max_vthread_extent 제약 목록."""
        limit = self.hw['max_vthread_extent']
        constraints = []
        for sid, iid, ext in self.s.get_vthread_extents():
            constraints.append({
                'stage_id': sid,
                'iter_id': iid,
                'sym_extent': ext,
                'limit': limit,
                'desc': f"max_vthread s{sid}.i{iid} ({self.s.stages[sid].op_name}): "
                        f"extent ≤ {limit}",
            })
        return constraints

    def check_vthread(self, sym_map=None):
        """vthread 제약 위반 목록."""
        if sym_map is None:
            sym_map = self.s.sym_map
        violations = []
        for c in self.build_vthread_constraints():
            val = eval_sym_extent(c['sym_extent'], sym_map)
            if isinstance(val, int) and val > c['limit']:
                violations.append(f"{c['desc']}: actual={val}")
        return violations

    # ─── 6) Max innermost split factor constraint ───
    def build_innermost_split_constraints(self):
        """각 SplitStep의 마지막 factor ≤ max_innermost_split_factor 제약 목록.
        마지막 factor = sp_{step_idx}_{last_length_idx}."""
        limit = self.hw['max_innermost_split_factor']
        sp_groups = self.pm._build_sp_groups()
        constraints = []
        for step_idx, names in sorted(sp_groups.items()):
            # 마지막 length의 sym_name
            last_name = names[-1]
            constraints.append({
                'sym_name': last_name,
                'step_idx': step_idx,
                'limit': limit,
                'desc': f"max_innermost_split {last_name}: value ≤ {limit}",
            })
        return constraints

    def check_innermost_split(self, sym_map=None):
        """innermost split factor 제약 위반 목록."""
        if sym_map is None:
            sym_map = self.s.sym_map
        violations = []
        for c in self.build_innermost_split_constraints():
            val = sym_map.get(c['sym_name'])
            if val is not None and isinstance(val, int) and val > c['limit']:
                violations.append(f"{c['desc']}: actual={val}")
        return violations

    # ─── 통합 체크 ───
    def check_all(self, sym_map=None):
        """모든 제약 위반을 한 번에 검사. 위반 목록 반환 (빈 리스트 = 모두 통과)."""
        violations = []
        violations.extend(self.check_vectorize(sym_map))
        violations.extend(self.check_shared_memory(sym_map))
        violations.extend(self.check_max_threads(sym_map))
        violations.extend(self.check_min_thread(sym_map))
        violations.extend(self.check_vthread(sym_map))
        violations.extend(self.check_innermost_split(sym_map))
        return violations

    # ═══════════════════════════════════════════════════════════
    # 2) 전처리: 초기 도메인 + 제약 파싱 + 변수 순서
    # ═══════════════════════════════════════════════════════════

    def _preprocess(self):
        """생성기 초기화 시 1회 실행.

        1) 초기 도메인: 약수 후보 + max_innermost_split 반영
        2) 제약 전처리: 표현식 트리 파싱, 변수 집합, RHS 저장
        3) 변수 순서: shared → thread → 나머지 (비선형/등장횟수 기준)
        """
        import re

        sp_groups = self.pm._build_sp_groups()
        sp_extents = self.pm._build_sp_extents(sp_groups)

        # ──── 1) 초기 도메인 구성 ────
        innermost_limit = self.hw['max_innermost_split_factor']
        innermost_names = set()
        for step_idx, names in sp_groups.items():
            innermost_names.add(names[-1])

        self._sp_groups = sp_groups
        self._sp_extents = sp_extents
        self._ur_names = [n for n in self.s.sym_map if n.startswith("ur_")]
        self._all_sp_names = []  # 모든 SP 이름 (step 순서)
        for step_idx in sorted(sp_groups.keys()):
            self._all_sp_names.extend(sp_groups[step_idx])

        # 각 SP 변수의 초기 도메인 = 약수 리스트
        # 단, 도메인 계산은 그룹 내 선행 변수의 값이 결정된 뒤에야 가능하므로,
        # 여기서는 extent만 기록하고, 도메인은 randomize_params 시 동적 계산
        # 다만 innermost 제약은 그룹 내 마지막이므로 정적 상한으로 기록
        self._innermost_names = innermost_names

        # ──── 2) 제약식 파싱 ────
        self._constraints = []
        self._var_constraints = {}  # {var_name: [constraint indices]}

        def _add_constraint(expr_tree, rhs, kind, desc, is_upper=True):
            """제약식 등록. is_upper=True: LHS ≤ RHS, False: LHS ≥ RHS."""
            idx = len(self._constraints)
            vars_in = expr_tree.variables()
            has_nonlinear = self._has_nonlinear(expr_tree)
            self._constraints.append({
                'tree': expr_tree,
                'rhs': rhs,
                'vars': vars_in,
                'kind': kind,
                'desc': desc,
                'is_upper': is_upper,
                'has_nonlinear': has_nonlinear,
            })
            for v in vars_in:
                self._var_constraints.setdefault(v, []).append(idx)

        # (a) vectorize: extent * dtype_bytes ≤ max_vector_bytes
        for c in self.build_vectorize_constraints():
            tree = _parse_expr_tree(str(c['sym_extent']))
            if c['dtype_bytes'] != 1:
                tree = ScaleMulNode(tree, c['dtype_bytes'])
            _add_constraint(tree, c['limit'], 'vectorize', c['desc'], is_upper=True)

        # (b) shared memory: sum(extent_i * dtype_bytes_i) ≤ limit
        sm = self.build_shared_memory_constraints()
        if sm['items']:
            children = []
            for item in sm['items']:
                tree = _parse_expr_tree(str(item['sym_extent']))
                if item['dtype_bytes'] != 1:
                    tree = ScaleMulNode(tree, item['dtype_bytes'])
                children.append(tree)
            sum_tree = SumNode(children) if len(children) > 1 else children[0]
            _add_constraint(sum_tree, sm['limit'], 'shared_memory', sm['desc'], is_upper=True)

        # (c) max_threads: extent ≤ max_threads_per_block
        for c in self.build_max_threads_constraints():
            tree = _parse_expr_tree(str(c['sym_extent']))
            _add_constraint(tree, c['limit'], 'max_threads', c['desc'], is_upper=True)

        # (d) min_thread: extent ≥ warp_size
        for c in self.build_min_thread_constraints():
            tree = _parse_expr_tree(str(c['sym_extent']))
            _add_constraint(tree, c['min_val'], 'min_thread', c['desc'], is_upper=False)

        # (e) vthread: extent ≤ max_vthread_extent
        for c in self.build_vthread_constraints():
            tree = _parse_expr_tree(str(c['sym_extent']))
            _add_constraint(tree, c['limit'], 'vthread', c['desc'], is_upper=True)

        # (f) innermost split: sp_X_last ≤ max_innermost_split_factor
        # 이미 초기 도메인에서 반영하므로 별도 제약으로 등록 안 함
        # (초기 도메인 상한 컷으로 충분)

        # ──── 3) 변수 할당 순서 결정 ────
        self._compute_var_order()

    @staticmethod
    def _has_nonlinear(node):
        """표현식 트리에 비선형 연산(min, ceil)이 포함되어 있는지 확인."""
        if isinstance(node, (MinNode, CeilDivNode)):
            return True
        if isinstance(node, (MulNode, AddNode, SubNode)):
            return ScheduleGenerator._has_nonlinear(node.left) or ScheduleGenerator._has_nonlinear(node.right)
        if isinstance(node, (ScaleMulNode,)):
            return ScheduleGenerator._has_nonlinear(node.child)
        if isinstance(node, SumNode):
            return any(ScheduleGenerator._has_nonlinear(c) for c in node.children)
        return False

    def _compute_var_order(self):
        """변수 할당 순서를 결정한다.

        우선순위:
          1) shared_memory 제약에만 등장하는 변수 먼저
          2) thread 관련 (max_threads, min_thread) 제약 변수
          3) 나머지 (vectorize, vthread 등)

        동일 우선순위 내에서:
          1) 비선형 연산이 없는 제약식의 변수 우선
          2) 현재 + 다른 미처리 제약식에서 등장 횟수 합이 큰 변수 우선

        단, SP 그룹 내 변수는 length_idx 순서를 유지해야 하므로
        그룹 단위로 우선순위를 결정한 뒤, 그룹 내부는 원래 순서 유지.
        """
        # 각 변수가 참여하는 제약 종류 분류
        shared_vars = set()
        thread_vars = set()
        other_vars = set()

        for ci, c in enumerate(self._constraints):
            kind = c['kind']
            vs = c['vars']
            if kind == 'shared_memory':
                shared_vars.update(vs)
            elif kind in ('max_threads', 'min_thread'):
                thread_vars.update(vs)
            else:
                other_vars.update(vs)

        # 각 SP 그룹에 우선순위 부여
        # 그룹 우선순위: 0 = shared, 1 = thread, 2 = other
        # 세부 우선순위: (비선형 여부, -등장횟수 합)

        # 변수별 전체 등장 횟수
        var_freq = {}
        for v in self._all_sp_names:
            var_freq[v] = len(self._var_constraints.get(v, []))

        group_priority = {}
        for step_idx, group in self._sp_groups.items():
            group_set = set(group)

            # 그룹이 어떤 제약 종류에 속하는지 판별
            in_shared = bool(group_set & shared_vars)
            in_thread = bool(group_set & thread_vars)

            if in_shared:
                cat = 0
            elif in_thread:
                cat = 1
            else:
                cat = 2

            # 그룹 내 변수가 참여하는 제약들의 비선형 여부 최소값
            min_nonlinear = True
            total_freq = 0
            for v in group:
                total_freq += var_freq.get(v, 0)
                for ci in self._var_constraints.get(v, []):
                    if not self._constraints[ci]['has_nonlinear']:
                        min_nonlinear = False

            group_priority[step_idx] = (cat, min_nonlinear, -total_freq)

        # 정렬 (extent가 없는 그룹은 마지막)
        sorted_steps = sorted(
            self._sp_groups.keys(),
            key=lambda si: group_priority.get(si, (3, True, 0))
        )

        self._var_order = []
        for step_idx in sorted_steps:
            self._var_order.extend(self._sp_groups[step_idx])

    # ═══════════════════════════════════════════════════════════
    # 3) 제약 만족 파라미터 생성
    # ═══════════════════════════════════════════════════════════

    def randomize_params(self, rng=None, max_retries=100):
        """모든 HW 제약을 만족하는 파라미터를 약수 조건 + HW 제약 하에 랜덤 생성.

        알고리즘:
          1) 모든 SP를 초기값(1)으로 설정, domains = {var: full divisor list}
          2) 변수 순서대로 하나씩 할당:
             a) 약수 조건으로 후보 리스트 구성
             b) innermost 제약 반영 (정적 상한)
             c) 관련 제약식에 대해 LHS_min > RHS 검사로 후보 필터링
             d) 남은 후보 중 랜덤 선택
          3) 상태 업데이트: 선택된 변수가 포함된 제약식의 도메인만 갱신

        Args:
            rng: random.Random 인스턴스 또는 None
            max_retries: 사후 rejection 최대 재시도 횟수

        Returns:
            dict: 설정된 {sym_name: value} 매핑

        Raises:
            RuntimeError: max_retries 초과 시
        """
        import random as _random
        if rng is None:
            rng = _random.Random()

        innermost_limit = self.hw['max_innermost_split_factor']

        violations = None
        for attempt in range(max_retries):
            result = {}
            # 모든 SP를 1로 초기화
            for name in self._all_sp_names:
                self.s.sym_map[name] = 1

            # 도메인 상태: {var_name: int (확정값) or None (미결정)}
            domains = {}
            for name in self._all_sp_names:
                domains[name] = None  # 미결정

            # 그룹별 remaining extent 추적
            group_remaining = {}
            for step_idx, ext in self._sp_extents.items():
                group_remaining[step_idx] = ext

            ok = True
            for name in self._var_order:
                parts = name.split("_")
                step_idx = int(parts[1])
                length_idx = int(parts[2])

                extent = self._sp_extents.get(step_idx)
                if extent is None:
                    # extent 없는 SP: 원본 값 유지
                    result[name] = self.s.sym_map[name]
                    domains[name] = self.s.sym_map[name]
                    continue

                remaining = group_remaining.get(step_idx, extent)
                candidates = self.pm._divisors(remaining)

                # 1) innermost 제약: 그룹의 마지막 변수 ≤ limit
                if name in self._innermost_names:
                    candidates = [c for c in candidates if c <= innermost_limit]

                # 2) 관련 제약식에 대해 LHS_min > RHS 검사
                constraint_indices = self._var_constraints.get(name, [])
                if constraint_indices:
                    candidates = self._filter_by_constraints(
                        name, candidates, constraint_indices, domains)

                if not candidates:
                    candidates = [1]

                chosen = rng.choice(candidates)
                self.s.sym_map[name] = chosen
                result[name] = chosen
                domains[name] = chosen

                # 그룹 remaining 갱신
                group_remaining[step_idx] = (remaining + chosen - 1) // chosen

            # unroll
            for name in self._ur_names:
                chosen = rng.choice(self.pm.UNROLL_CANDIDATES)
                self.s.sym_map[name] = chosen
                result[name] = chosen

            # 사후 검증
            violations = self.check_all()
            if not violations:
                return result

        raise RuntimeError(
            f"Failed to find valid params after {max_retries} retries. "
            f"Last violations: {violations}")

    def _filter_by_constraints(self, var_name, candidates, constraint_indices, domains):
        """후보 리스트에서 제약을 위반하는 값을 제거.

        각 후보 v에 대해:
          1) domains[var_name] = v 로 가정
          2) 관련 제약식의 LHS interval을 계산
          3) upper 제약: LHS_min > RHS → 제거
             lower 제약: LHS_max < RHS → 제거

        이진 검색 최적화: upper 제약에서 후보가 정렬 상태이고
        LHS가 var_name에 대해 단조증가이면, 최대 유효 인덱스를 이진 검색으로 탐색.

        Args:
            var_name: 현재 결정 중인 SP 이름
            candidates: 정렬된 후보 리스트
            constraint_indices: 관련 제약 인덱스 리스트
            domains: 현재 도메인 상태

        Returns:
            list: 유효한 후보 리스트
        """
        if not candidates:
            return candidates

        # 도메인 스냅샷 (interval 계산용): 확정된 변수는 int, 미확정은 리스트
        # 미확정 변수의 후보 리스트는 비용이 크므로, [1, max_extent] 근사 사용
        interval_domains = {}
        for v, d in domains.items():
            if d is not None:
                interval_domains[v] = d  # 확정: int → (d, d)
            else:
                # 미확정: 도메인의 범위 = [1, extent 전체]
                # (정확한 약수 리스트를 구하면 비용이 크므로 상한을 extent로 근사)
                vparts = v.split("_")
                vstep = int(vparts[1])
                vext = self._sp_extents.get(vstep)
                interval_domains[v] = [1, vext] if vext else 1

        # 후보 필터링: 이진 검색 + 선형 검사 혼합
        # upper 제약 (LHS ≤ RHS): 이진 검색 가능 (후보 증가 → LHS_min 증가)
        # lower 제약 (LHS ≥ RHS): 이진 검색 가능 (후보 증가 → LHS_max 증가)
        upper_constraints = []
        lower_constraints = []
        for ci in constraint_indices:
            c = self._constraints[ci]
            if c['is_upper']:
                upper_constraints.append(c)
            else:
                lower_constraints.append(c)

        # ─── upper 제약: 이진 검색으로 최대 유효 인덱스 ───
        max_valid_idx = len(candidates) - 1
        for c in upper_constraints:
            idx = self._bisect_upper(var_name, candidates, c, interval_domains)
            max_valid_idx = builtins_min(max_valid_idx, idx)

        # ─── lower 제약: 이진 검색으로 최소 유효 인덱스 ───
        min_valid_idx = 0
        for c in lower_constraints:
            idx = self._bisect_lower(var_name, candidates, c, interval_domains)
            min_valid_idx = max(min_valid_idx, idx)

        if min_valid_idx > max_valid_idx:
            return [candidates[0]]

        return candidates[min_valid_idx:max_valid_idx + 1]

    def _bisect_upper(self, var_name, candidates, constraint, interval_domains):
        """upper 제약 (LHS ≤ RHS)에 대해 유효한 최대 후보 인덱스를 이진 검색.

        후보가 커질수록 LHS_min이 증가하는 단조성을 이용.
        LHS_min > RHS 이면 해당 후보 이상은 모두 무효.
        """
        tree = constraint['tree']
        rhs = constraint['rhs']

        lo, hi = 0, len(candidates) - 1

        # 빠른 경로: 최대 후보 검사
        test_dom = dict(interval_domains)
        test_dom[var_name] = candidates[hi]
        lhs_min, _ = tree.interval(test_dom)
        if lhs_min <= rhs:
            return hi  # 모두 유효

        # 빠른 경로: 최소 후보 검사
        test_dom[var_name] = candidates[lo]
        lhs_min, _ = tree.interval(test_dom)
        if lhs_min > rhs:
            return -1  # 모두 무효 (candidates[0] 이라도 반환하게 됨)

        # 이진 검색
        while lo < hi:
            mid = (lo + hi + 1) // 2
            test_dom[var_name] = candidates[mid]
            lhs_min, _ = tree.interval(test_dom)
            if lhs_min <= rhs:
                lo = mid
            else:
                hi = mid - 1

        return lo

    def _bisect_lower(self, var_name, candidates, constraint, interval_domains):
        """lower 제약 (LHS ≥ RHS)에 대해 유효한 최소 후보 인덱스를 이진 검색.

        후보가 커질수록 LHS_max가 증가하는 단조성을 이용.
        LHS_max < RHS 이면 해당 후보 이하는 모두 무효.
        """
        tree = constraint['tree']
        rhs = constraint['rhs']

        lo, hi = 0, len(candidates) - 1

        # 빠른 경로: 최소 후보 검사
        test_dom = dict(interval_domains)
        test_dom[var_name] = candidates[lo]
        _, lhs_max = tree.interval(test_dom)
        if lhs_max >= rhs:
            return lo  # 모두 유효

        # 빠른 경로: 최대 후보 검사
        test_dom[var_name] = candidates[hi]
        _, lhs_max = tree.interval(test_dom)
        if lhs_max < rhs:
            return hi + 1  # 모두 무효

        # 이진 검색
        while lo < hi:
            mid = (lo + hi) // 2
            test_dom[var_name] = candidates[mid]
            _, lhs_max = tree.interval(test_dom)
            if lhs_max >= rhs:
                hi = mid
            else:
                lo = mid + 1

        return lo


# ─────────────────────────────────────────────────────────────
#  편의 함수: SymbolicState 생성 + TransformApplier 적용 (원스텝)
# ─────────────────────────────────────────────────────────────
def build_symbolic_state(compute_dag, state):
    """compute_dag + state로부터 SymbolicState를 생성하고 steps를 적용.
    Returns: SymbolicState (apply 완료)
    """
    sym = SymbolicState(compute_dag)
    applier = TransformApplier(sym)
    applier.apply_steps(state)
    return sym

In [76]:
# ─────────────────────────────────────────────────────────────
#  검증 유틸리티
# ─────────────────────────────────────────────────────────────


def verify_symbolic_state(task, state, verbose=False):
    """
    SymbolicState를 생성·적용한 뒤 실제 InferBound된 state와 비교.
    Returns: (ok: bool, summary: str, sym_state: SymbolicState)
    """
    sym_state = build_symbolic_state(task.compute_dag, state)
    bounded = task.compute_dag.infer_bound_from_state(state)

    stage_mismatch = []
    name_mm = 0
    ann_mm = 0
    ext_mm = 0
    ext_total = 0
    details = []

    n_stages = min(len(bounded.stages), len(sym_state.stages))
    if len(bounded.stages) != len(sym_state.stages):
        details.append(f"Stage count: real={len(bounded.stages)} sym={len(sym_state.stages)}")

    for sid in range(n_stages):
        rs = bounded.stages[sid]
        ss = sym_state.stages[sid]
        if len(rs.iters) != len(ss.iters):
            stage_mismatch.append((sid, len(rs.iters), len(ss.iters)))
            continue
        for iid in range(len(rs.iters)):
            ri, si = rs.iters[iid], ss.iters[iid]
            if str(ri.name) != si.name:
                name_mm += 1
                if verbose and name_mm <= 5:
                    details.append(f"  NAME s{sid}.i{iid}: real='{ri.name}' sym='{si.name}'")
            if int(ri.annotation) != si.annotation:
                ann_mm += 1
                if verbose and ann_mm <= 5:
                    details.append(f"  ANN  s{sid}.i{iid}: real={int(ri.annotation)} sym={si.annotation}")
            re_ext = int(ri.range.extent) if ri.range is not None else None
            se_ext = eval_sym_extent(si.extent, sym_state.sym_map)
            ext_total += 1
            # CA tightening으로 인해 symbolic extent(upper bound)가 실제보다 클 수 있음.
            # se_ext >= re_ext이면 OK. (CA가 없는 iter는 정확히 일치)
            if re_ext is not None and se_ext is not None:
                if se_ext < re_ext:
                    ext_mm += 1
                    if verbose and ext_mm <= 5:
                        details.append(f"  EXT  s{sid}.i{iid}('{si.name}'): real={re_ext} sym={si.extent}→eval={se_ext}")
            elif re_ext != se_ext:
                ext_mm += 1
                if verbose and ext_mm <= 5:
                    details.append(f"  EXT  s{sid}.i{iid}('{si.name}'): real={re_ext} sym={si.extent}→eval={se_ext}")

    ok = (len(stage_mismatch) == 0 and name_mm == 0 and ann_mm == 0 and ext_mm == 0
          and len(bounded.stages) == len(sym_state.stages))
    parts = []
    if stage_mismatch:
        parts.append(f"iter_count_mm={len(stage_mismatch)}")
    if name_mm:
        parts.append(f"name_mm={name_mm}")
    if ann_mm:
        parts.append(f"ann_mm={ann_mm}")
    if ext_mm:
        parts.append(f"ext_mm={ext_mm}/{ext_total}")
    summary = "PASS" if ok else "FAIL(" + ", ".join(parts) + ")"
    if verbose and details:
        summary += "\n" + "\n".join(details)
    return ok, summary, sym_state

In [46]:
task_idx = 0
task = tasks[task_idx]
policy = search_policies[task_idx]
states = policy.sample_initial_population()
sym_state = build_symbolic_state(task.compute_dag, states[0])

print("=== vectorize extents ===")
for sid, iid, ext in sym_state.get_vectorize_extents():
    print(f"  s{sid}.i{iid} ({sym_state.stages[sid].op_name}): {ext}")

print("\n=== thread extents ===")
for sid, iid, ext in sym_state.get_thread_extents():
    ann_str = ANNOTATION_STR[sym_state.stages[sid].iters[iid].annotation]
    print(f"  s{sid}.i{iid} ({sym_state.stages[sid].op_name}) [{ann_str}]: {ext}")

print("\n=== vthread extents ===")
for sid, iid, ext in sym_state.get_vthread_extents():
    print(f"  s{sid}.i{iid} ({sym_state.stages[sid].op_name}): {ext}")

print("\n=== shared memory extents ===")
for sid, op_name, ext in sym_state.get_shared_memory_extents():
    val = eval_sym_extent(ext, sym_state.sym_map)
    print(f"  s{sid} ({op_name}): {ext}  →  eval={val}")

Generate Sketches		#s: 1
Sample Initial Population	#s: 70	fail_ct: 1978	Time elapsed: 4.66
=== vectorize extents ===
  s5.i2 (data_pack.shared): min(sp_56_0,sp_10_0*sp_10_1*sp_10_2*sp_10_3*sp_11_0*sp_11_1*sp_11_2*sp_11_3*sp_12_0*sp_12_1*sp_12_2*sp_12_3*sp_14_0*sp_14_1)
  s7.i2 (p1.shared): min(sp_51_0,sp_10_0*sp_10_1*sp_10_2*sp_10_3*sp_11_0*sp_11_1*sp_11_2*sp_11_3*sp_13_0*sp_13_1*sp_13_2*sp_13_3*sp_14_0*sp_14_1)

=== thread extents ===
  s4.i1 (data_pack) [threadIdx.x]: min(sp_61_0,ceil(196/(min(sp_28_0,196)))*ceil(64/(min(sp_29_0,64)))*min(sp_28_0,196)*min(sp_29_0,64))
  s5.i1 (data_pack.shared) [threadIdx.x]: sp_13_1*sp_12_1*sp_11_1*sp_10_1
  s7.i1 (p1.shared) [threadIdx.x]: sp_13_1*sp_12_1*sp_11_1*sp_10_1
  s9.i2 (bgemm) [threadIdx.x]: min(sp_10_1,ceil(6/(min(sp_10_2*sp_10_3,6))))*min(sp_11_1,ceil(6/(min(sp_11_2*sp_11_3,6))))*min(sp_12_1,ceil(196/(min(sp_12_2*sp_12_3,196))))*min(sp_13_1,ceil(64/(min(sp_13_2*sp_13_3,64))))
  s11.i1 (inverse) [threadIdx.x]: min(sp_41_0,ceil(196/(min(s

In [105]:
"""
검증: 동일 task에서 step 시퀀스의 구조가 같고 split factor / unroll pragma 값만 다른
state들은 동일한 symbolic structure(sym_state 문자열)를 가져야 한다.

step fingerprint: split의 lengths/extent, pragma의 값($뒤)을 제외한 모든 구조 속성.
"""
import sys, io
from collections import defaultdict

def step_structural_fingerprint(step):
    """step의 구조적 속성만 추출하여 hashable tuple 반환.
    SplitStep의 lengths/extent, PragmaStep의 pragma 값은 제외."""
    tk = step.type_key.split(".")[-1]
    if tk == "AnnotationStep":
        return (tk, int(step.stage_id), int(step.iter_id), int(step.annotation))
    elif tk == "FuseStep":
        return (tk, int(step.stage_id), tuple(int(x) for x in step.fused_ids))
    elif tk == "PragmaStep":
        # pragma_type에서 값 부분 제거: "auto_unroll_max_step$16" → "auto_unroll_max_step"
        ptype = str(step.pragma_type).split("$")[0]
        return (tk, int(step.stage_id), int(step.iter_id), ptype)
    elif tk == "ReorderStep":
        return (tk, int(step.stage_id), tuple(int(x) for x in step.after_ids))
    elif tk == "SplitStep":
        # lengths 값 제외, 개수와 inner_to_outer만 포함
        return (tk, int(step.stage_id), int(step.iter_id),
                len(step.lengths), bool(step.inner_to_outer))
    elif tk == "FollowSplitStep":
        return (tk, int(step.stage_id), int(step.iter_id),
                int(step.src_step_id), int(step.n_split))
    elif tk == "FollowFusedSplitStep":
        return (tk, int(step.stage_id), int(step.iter_id),
                tuple(int(x) for x in step.src_step_ids),
                int(step.level), bool(step.factor_or_nparts))
    elif tk == "StorageAlignStep":
        # factor/offset는 구조적 속성이므로 포함
        return (tk, int(step.stage_id), int(step.iter_id),
                int(step.factor), int(step.offset))
    elif tk == "ComputeAtStep":
        return (tk, int(step.stage_id), int(step.target_stage_id), int(step.target_iter_id))
    elif tk == "ComputeInlineStep":
        return (tk, int(step.stage_id))
    elif tk == "ComputeRootStep":
        return (tk, int(step.stage_id))
    elif tk == "CacheReadStep":
        return (tk, int(step.stage_id), str(step.scope_name),
                tuple(int(x) for x in step.reader_stage_ids))
    elif tk == "CacheWriteStep":
        return (tk, int(step.stage_id), str(step.scope_name))
    else:
        return (tk, int(step.stage_id))


def state_structural_fingerprint(state):
    """state의 전체 step 시퀀스에서 구조적 fingerprint를 추출."""
    return tuple(step_structural_fingerprint(s) for s in state.transform_steps)


def sym_state_structure_str(sym_state):
    """SymbolicState의 구조 문자열: sym_map의 key→value 매핑을 제거하고
    symbolic expression 자체만 비교 (= sym_name이 같으면 같은 구조)."""
    return sym_state.to_str(delete_trivial_loop=False)


# ══════════════════════════════════════════════════════════════
#  검증 실행
# ══════════════════════════════════════════════════════════════
total_groups = 0
total_mismatches = 0
total_states_checked = 0

for task_idx in range(len(tasks)):
    task = tasks[task_idx]
    policy = search_policies[task_idx]

    old_stderr = sys.stderr
    sys.stderr = io.StringIO()
    try:
        states = policy.sample_initial_population()
    finally:
        sys.stderr = old_stderr

    # fingerprint별로 state 그룹핑
    groups = defaultdict(list)
    for si, state in enumerate(states):
        fp = state_structural_fingerprint(state)
        groups[fp].append((si, state))

    task_groups = 0
    task_mismatches = 0

    for fp, group in groups.items():
        if len(group) < 2:
            continue  # 비교 대상 없음
        task_groups += 1

        # 첫 번째 state의 sym_state를 reference로 사용
        ref_si, ref_state = group[0]
        ref_sym = build_symbolic_state(task.compute_dag, ref_state)
        ref_str = sym_state_structure_str(ref_sym)

        for si, state in group[1:]:
            total_states_checked += 1
            cur_sym = build_symbolic_state(task.compute_dag, state)
            cur_str = sym_state_structure_str(cur_sym)

            if cur_str != ref_str:
                task_mismatches += 1
                if task_mismatches <= 2:
                    print(f"\n❌ MISMATCH T{task_idx} state[{ref_si}] vs state[{si}]")
                    # 차이 찾기
                    ref_lines = ref_str.split("\n")
                    cur_lines = cur_str.split("\n")
                    for li in range(max(len(ref_lines), len(cur_lines))):
                        rl = ref_lines[li] if li < len(ref_lines) else "<missing>"
                        cl = cur_lines[li] if li < len(cur_lines) else "<missing>"
                        if rl != cl:
                            print(f"  line {li}: ref='{rl}'")
                            print(f"  line {li}: cur='{cl}'")

    total_groups += task_groups
    total_mismatches += task_mismatches

    status = "✅" if task_mismatches == 0 else "❌"
    n_multi = sum(1 for g in groups.values() if len(g) >= 2)
    print(f"T{task_idx:2d} {status} {len(states):3d} states, "
          f"{len(groups):2d} unique fps, {n_multi:2d} multi-state groups, "
          f"{task_mismatches} mismatches  [{task.desc[:55]}]")

print()
print("=" * 70)
print(f"Total: {total_states_checked} pairwise comparisons across {total_groups} groups")
if total_mismatches == 0:
    print("✅✅✅ ALL SAME-STRUCTURE STATES HAVE IDENTICAL sym_state! ✅✅✅")
else:
    print(f"❌ {total_mismatches} mismatches found")
print("=" * 70)

Generate Sketches		#s: 1
Sample Initial Population	#s: 63	fail_ct: 1985	Time elapsed: 5.03
Sample Initial Population	#s: 63	fail_ct: 1985	Time elapsed: 5.03
T 0 ✅  63 states,  1 unique fps,  1 multi-state groups, 0 mismatches  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
Generate Sketches		#s: 1
T 0 ✅  63 states,  1 unique fps,  1 multi-state groups, 0 mismatches  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
Generate Sketches		#s: 1
Sample Initial Population	#s: 81	fail_ct: 1967	Time elapsed: 5.27
Sample Initial Population	#s: 81	fail_ct: 1967	Time elapsed: 5.27
T 1 ✅  81 states,  1 unique fps,  1 multi-state groups, 0 mismatches  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
Generate Sketches		#s: 1
T 1 ✅  81 states,  1 unique fps,  1 multi-state groups, 0 mismatches  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
Generate Sketches		#s: 1
Sample Initial Population	#s: 78	fail_ct: 1970	Time elapsed: 5.04
Sample Initial Population	#s: 78	

In [35]:
# ═══════════════════════════════════════════════════════════════
# 종합 검증: 모든 task × 여러 state에서 SymbolicState 검증
# ═══════════════════════════════════════════════════════════════
import time


total_pass = 0
total_fail = 0
fail_details = []

t0 = time.time()

for task_idx in range(len(tasks)):
    print(f"Testing Task {task_idx:2d}/{len(tasks)-1}...")
    task = tasks[task_idx]
    policy = search_policies[task_idx]

    try:
        init_states = policy.sample_initial_population()
        states = policy.evolutionary_search(init_states, 1000)
    except Exception as e:
        print(f"T{task_idx:2d} [{task.desc[:50]:50s}]  ⚠️  sample failed: {e}")
        continue

    task_pass = 0
    task_fail = 0
    task_fail_msgs = []

    for si in range(len(states)):
        state = states[si]
        try:
            ok, summary, sym_state = verify_symbolic_state(task, state, verbose=True)
        except Exception as e:
            ok = False
            summary = f"EXCEPTION: {e}"

        if ok:
            task_pass += 1
        else:
            task_fail += 1
            task_fail_msgs.append(f"    state[{si}]: {summary}")

    total_pass += task_pass
    total_fail += task_fail

    status = "✅" if task_fail == 0 else "❌"
    print(f"T{task_idx:2d} {status} {task_pass}/{len(states)} pass  [{task.desc[:60]}]")
    if task_fail_msgs:
        for msg in task_fail_msgs:
            print(msg)
        fail_details.extend(task_fail_msgs)

elapsed = time.time() - t0

print()
print("=" * 70)
print(f"Total: {total_pass + total_fail} tests, "
      f"{total_pass} passed, {total_fail} failed  "
      f"({elapsed:.1f}s)")
if total_fail == 0:
    print("✅✅✅ ALL TESTS PASSED! ✅✅✅")
else:
    print(f"❌ {total_fail} failures")
print("=" * 70)

Testing Task  0/23...
Generate Sketches		#s: 1
Sample Initial Population	#s: 71	fail_ct: 1977	Time elapsed: 5.19
GA Iter: 0	Max score: 0.9940	Min score: 0.0028	#Pop: 71	#M+: 0	#M-: 0
GA Iter: 4	Max score: 0.9998	Min score: 0.7604	#Pop: 1000	#M+: 1400	#M-: 0
EvolutionarySearch		#s: 1000	Time elapsed: 72.82
T 0 ✅ 1000/1000 pass  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_trans]
Testing Task  1/23...
Generate Sketches		#s: 1
Sample Initial Population	#s: 88	fail_ct: 1960	Time elapsed: 5.02
GA Iter: 0	Max score: 0.9905	Min score: 0.0259	#Pop: 88	#M+: 0	#M-: 0
GA Iter: 4	Max score: 0.9992	Min score: 0.7796	#Pop: 1000	#M+: 1400	#M-: 0
EvolutionarySearch		#s: 1000	Time elapsed: 72.96
T 1 ✅ 1000/1000 pass  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_trans]
Testing Task  2/23...
Generate Sketches		#s: 1
Sample Initial Population	#s: 81	fail_ct: 1967	Time elapsed: 4.93
GA Iter: 0	Max score: 0.9995	Min score: 0.0040	#Pop: 81	#M+: 0	#M-: 0
GA Iter: 4	Max score: 0.9996	Min 

In [102]:
# ═══════════════════════════════════════════════════════════════
# JSON에서 스케줄 로드 → SymbolicState 생성 → 파라미터 조회/삽입
# ═══════════════════════════════════════════════════════════════
import json
from collections import defaultdict

JSON_PATH = "/root/work/tvm-ansor/gallery/logs_json/resnet_18/resnet_18-B1.json"

# ─────────────────────────────────────────────
# 1) JSON → RecordReader로 (MeasureInput, MeasureResult) 로드
# ─────────────────────────────────────────────
def load_records_from_json(json_path):
    """RecordReader로 JSON 파일의 모든 레코드를 로드.
    Returns: list of (MeasureInput, MeasureResult)
    """
    reader = auto_scheduler.RecordReader(json_path)
    inputs, results = reader.read_lines()
    return list(zip(inputs, results))

records = load_records_from_json(JSON_PATH)
print(f"Total records loaded: {len(records)}")

# ─────────────────────────────────────────────
# 2) workload_key별 그룹화 → task 매칭
# ─────────────────────────────────────────────
def group_records_by_task(records, tasks):
    """레코드를 workload_key 기준으로 task와 매칭.
    Returns: dict {task_idx: [(MeasureInput, MeasureResult), ...]}
    """
    wkey_to_tidx = {}
    for tidx, task in enumerate(tasks):
        wkey_to_tidx[task.workload_key] = tidx

    grouped = defaultdict(list)
    unmatched = 0
    for inp, res in records:
        wkey = inp.task.workload_key
        tidx = wkey_to_tidx.get(wkey)
        if tidx is not None:
            grouped[tidx].append((inp, res))
        else:
            unmatched += 1

    print(f"Matched tasks: {len(grouped)}, Unmatched records: {unmatched}")
    for tidx in sorted(grouped.keys()):
        recs = grouped[tidx]
        best = min(recs, key=lambda x: x[1].costs[0] if x[1].costs else float('inf'))
        best_lat = float(best[1].costs[0]) * 1000 if best[1].costs else None
        print(f"  T{tidx:2d}: {len(recs):4d} records, "
              f"best={best_lat:.3f}ms  [{tasks[tidx].desc[:55]}]")
    return grouped

grouped = group_records_by_task(records, tasks)

# ─────────────────────────────────────────────
# 3) 특정 task/record에서 SymbolicState 생성
# ─────────────────────────────────────────────
def create_sym_state_from_record(task, measure_input):
    """MeasureInput에서 state를 추출하여 SymbolicState 생성.
    Returns: (SymbolicState, state_object)
    """
    state = measure_input.state
    sym = build_symbolic_state(task.compute_dag, state)
    return sym, state

# 테스트: 첫 번째 매칭 task의 best record
first_tidx = sorted(grouped.keys())[0]
first_records = grouped[first_tidx]
best_rec = min(first_records, key=lambda x: x[1].costs[0] if x[1].costs else float('inf'))
task = tasks[first_tidx]

sym_state, state_obj = create_sym_state_from_record(task, best_rec[0])
print(f"\n{'='*70}")
print(f"Task {first_tidx}: {task.desc[:70]}")
print(f"{'='*70}")

# ─────────────────────────────────────────────
# 4) 파라미터 조회 (약수 후보 포함)
# ─────────────────────────────────────────────
pm = SymParamManager(sym_state)

print(f"\n[파라미터 요약]")
print(pm.get_sym_params_summary())

print(f"\n[파라미터 상세 - 약수 후보]")
params = pm.get_sym_params()
for p in params:
    kind_str = "SP" if p["kind"] == "sp" else "UR"
    cands = p["candidates"]
    cands_str = str(cands) if len(cands) <= 15 else f"[{cands[0]}..{cands[-1]}] ({len(cands)}개)"
    print(f"  {kind_str} {p['name']:12s} = {str(p['value']):>5s}  "
          f"extent={str(p['extent']):>5s}  candidates={cands_str}")

Total records loaded: 2408
Matched tasks: 24, Unmatched records: 0
  T 0:   72 records, best=0.031ms  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
  T 1:   76 records, best=0.034ms  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
  T 2:  136 records, best=0.049ms  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
  T 3:  140 records, best=0.054ms  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
  T 4:   72 records, best=0.033ms  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
  T 5:   72 records, best=0.029ms  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
  T 6:   72 records, best=0.042ms  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
  T 7:  136 records, best=0.050ms  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
  T 8:  136 records, best=0.029ms  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
  T 9:   72 records, best=0.034ms  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_]
  T

In [103]:
# ═══════════════════════════════════════════════════════════════
# ScheduleGenerator 테스트: 제약 만족 파라미터 생성
# ═══════════════════════════════════════════════════════════════
import random

HW_PARAM = {
    'max_vector_bytes' : 16,
    'max_shared_memory_per_block' : 49152,
    'max_threads_per_block' : 1024,
    'max_vthread_extent' : 8,
    'warp_size' : 32,
    'max_innermost_split_factor' : 64,
}

# ─── 1) 제약식 빌드 확인 ───
print("=" * 70)
print("1) 제약식 빌드 확인 (T0 best record)")
print("=" * 70)
sym_test, _ = create_sym_state_from_record(tasks[first_tidx], best_rec[0])
gen = ScheduleGenerator(sym_test, HW_PARAM)

print("\n[Vectorize constraints]")
for c in gen.build_vectorize_constraints():
    print(f"  {c['desc']}")

print("\n[Shared memory constraint]")
c_shared = gen.build_shared_memory_constraints()
print(f"  {c_shared['desc']}")
for item in c_shared['items']:
    val = eval_sym_extent(item['sym_extent'], sym_test.sym_map)
    print(f"    {item['op_name']}: extent={val}, dtype_bytes={item['dtype_bytes']}, bytes={val*item['dtype_bytes']}")

print("\n[Max threads constraints]")
for c in gen.build_max_threads_constraints():
    print(f"  {c['desc']}")

print("\n[Min thread constraints]")
for c in gen.build_min_thread_constraints():
    print(f"  {c['desc']}")

print("\n[Vthread constraints]")
for c in gen.build_vthread_constraints():
    print(f"  {c['desc']}")

print("\n[Innermost split constraints]")
for c in gen.build_innermost_split_constraints():
    print(f"  {c['desc']}")

# ─── 2) 원본 파라미터 제약 체크 ───
print("\n" + "=" * 70)
print("2) 원본 파라미터 제약 체크")
print("=" * 70)
violations = gen.check_all()
if violations:
    print(f"  ❌ {len(violations)} violations:")
    for v in violations:
        print(f"    {v}")
else:
    print("  ✅ 모든 제약 통과")

# ─── 3) constrained randomize_params 테스트 ───
print("\n" + "=" * 70)
print("3) constrained randomize_params 테스트")
print("=" * 70)

n_samples = 10
seed = 42
total_ok = 0
total_err = 0

for tidx in sorted(grouped.keys())[:6]:
    task = tasks[tidx]
    recs = grouped[tidx]
    best = min(recs, key=lambda x: x[1].costs[0] if x[1].costs else float('inf'))

    print(f"T{tidx:2d}: {task.desc[:55]}")

    for trial in range(n_samples):
        rng = random.Random(seed + trial)
        sym, st = create_sym_state_from_record(task, best[0])
        gen = ScheduleGenerator(sym, HW_PARAM)

        try:
            chosen = gen.randomize_params(rng=rng)
        except RuntimeError as e:
            total_err += 1
            print(f"  trial {trial}: ❌ RuntimeError: {e}")
            continue

        # 제약 재검증
        violations = gen.check_all()
        if violations:
            total_err += 1
            print(f"  trial {trial}: ❌ {violations}")
        else:
            total_ok += 1

    # 마지막 trial 결과 표시
    pm_last = SymParamManager(sym)
    print(f"  example: {pm_last.get_sym_params_summary()}")
    print()

print(f"{'='*60}")
print(f"Total: {total_ok + total_err} tests, {total_ok} passed, {total_err} failed")
if total_err == 0:
    print("✅ ALL PASSED — 모든 생성 파라미터가 HW 제약을 만족!")
else:
    print(f"❌ {total_err} failures")

1) 제약식 빌드 확인 (T0 best record)

[Vectorize constraints]
  vectorize s5.i2 (data_pack.shared): extent*4 ≤ 16
  vectorize s7.i2 (p1.shared): extent*4 ≤ 16

[Shared memory constraint]
  shared_memory: sum(extent*dtype_bytes) ≤ 49152
    data_pack.shared: extent=672, dtype_bytes=4, bytes=2688
    p1.shared: extent=768, dtype_bytes=4, bytes=3072

[Max threads constraints]
  max_threads s4.i1 (data_pack): extent ≤ 1024
  max_threads s5.i1 (data_pack.shared): extent ≤ 1024
  max_threads s7.i1 (p1.shared): extent ≤ 1024
  max_threads s9.i2 (bgemm): extent ≤ 1024
  max_threads s11.i1 (inverse): extent ≤ 1024
  max_threads s14.i1 (T_add): extent ≤ 1024

[Min thread constraints]

[Vthread constraints]
  max_vthread s9.i1 (bgemm): extent ≤ 8

[Innermost split constraints]
  max_innermost_split sp_3_0: value ≤ 64
  max_innermost_split sp_4_0: value ≤ 64
  max_innermost_split sp_10_3: value ≤ 64
  max_innermost_split sp_11_3: value ≤ 64
  max_innermost_split sp_12_3: value ≤ 64
  max_innermost_split 

In [83]:
# ═══ Rejection 원인 분석: 어떤 제약이 rejection을 유발하는가? ═══
import random, re
from collections import Counter, defaultdict

rng_base = random.Random(42)
n_trials = 500  # task별 시도 횟수

# 분석: 정적 상한 + vthread 예산 적용 후, check_all에서 어떤 제약이 실패하는가?
rejection_stats = Counter()  # "check_name" → 위반 횟수
total_attempts = 0
total_pass = 0
per_task_stats = {}

for tidx in sorted(grouped.keys()):
    task = tasks[tidx]
    recs = grouped[tidx]
    best = min(recs, key=lambda x: x[1].costs[0] if x[1].costs else float('inf'))

    sym, st = create_sym_state_from_record(task, best[0])
    gen = ScheduleGenerator(sym, HW_PARAM)

    task_total = 0
    task_pass_count = 0
    task_violations = Counter()

    for trial in range(n_trials):
        task_total += 1
        total_attempts += 1

        rng = random.Random(rng_base.randint(0, 10**9))
        try:
            gen.randomize_params(rng, max_retries=1)
            task_pass_count += 1
            total_pass += 1
        except RuntimeError:
            # 위반 확인 — 마지막 시도의 sym_map 상태에서 개별 제약 체크
            for name, checker in [
                ("vectorize", gen.check_vectorize),
                ("shared_memory", gen.check_shared_memory),
                ("max_threads", gen.check_max_threads),
                ("min_thread", gen.check_min_thread),
                ("vthread", gen.check_vthread),
                ("innermost", gen.check_innermost_split),
            ]:
                v = checker()
                if v:
                    task_violations[name] += 1
                    rejection_stats[name] += 1

    per_task_stats[tidx] = {
        'total': task_total, 'pass': task_pass_count,
        'rate': task_pass_count/task_total if task_total > 0 else 0,
        'violations': dict(task_violations),
    }

print(f"=== 전체 통계: {total_pass}/{total_attempts} 통과 ({100*total_pass/total_attempts:.1f}%) ===")
print(f"\n위반 유형별 빈도 (복수 위반 가능):")
for k, v in rejection_stats.most_common():
    pct = 100*v/(total_attempts - total_pass) if total_attempts > total_pass else 0
    print(f"  {k}: {v} ({pct:.1f}% of failures)")

print(f"\n=== Task별 상세 (실패 있는 것만) ===")
for tidx in sorted(per_task_stats.keys()):
    s = per_task_stats[tidx]
    if s['pass'] < s['total']:
        print(f"  T{tidx:2d}: {s['pass']}/{s['total']} ({100*s['rate']:.0f}%) violations={s['violations']}")


=== 전체 통계: 2780/12000 통과 (23.2%) ===

위반 유형별 빈도 (복수 위반 가능):
  shared_memory: 9067 (98.3% of failures)
  max_threads: 2002 (21.7% of failures)

=== Task별 상세 (실패 있는 것만) ===
  T 0: 24/500 (5%) violations={'shared_memory': 470, 'max_threads': 156}
  T 1: 51/500 (10%) violations={'shared_memory': 438, 'max_threads': 125}
  T 2: 42/500 (8%) violations={'shared_memory': 449, 'max_threads': 122}
  T 3: 38/500 (8%) violations={'shared_memory': 458, 'max_threads': 91}
  T 4: 33/500 (7%) violations={'shared_memory': 455, 'max_threads': 159}
  T 5: 28/500 (6%) violations={'shared_memory': 461, 'max_threads': 114}
  T 6: 40/500 (8%) violations={'shared_memory': 449, 'max_threads': 114}
  T 7: 28/500 (6%) violations={'shared_memory': 472, 'max_threads': 78}
  T 8: 30/500 (6%) violations={'shared_memory': 467, 'max_threads': 152}
  T 9: 44/500 (9%) violations={'shared_memory': 443, 'max_threads': 113}
  T10: 46/500 (9%) violations={'shared_memory': 446, 'max_threads': 113}
  T11: 37/500 (7%) violatio

In [85]:
# ═══ 제약 구조 분석: shared_memory / max_threads에 기여하는 SP 파라미터 ═══
import re

# Task 0 하나만 상세 분석
tidx = sorted(grouped.keys())[0]
task = tasks[tidx]
recs = grouped[tidx]
best = min(recs, key=lambda x: x[1].costs[0] if x[1].costs else float('inf'))
sym, st = create_sym_state_from_record(task, best[0])
gen = ScheduleGenerator(sym, HW_PARAM)

print(f"Task {tidx}: {task.desc[:60]}")

# Shared memory
c_shared = gen.build_shared_memory_constraints()
print(f"\n[Shared Memory] limit={c_shared['limit']}")
all_shared_sp = set()
for item in c_shared['items']:
    ext_str = str(item['sym_extent'])
    sp_names = sorted(set(re.findall(r'sp_\d+_\d+', ext_str)))
    all_shared_sp.update(sp_names)
    val = eval_sym_extent(item['sym_extent'], sym.sym_map)
    print(f"  {item['op_name']}: {ext_str}")
    print(f"    SPs: {sp_names}, val={val}, bytes={val*item['dtype_bytes']}")

# Max threads
print(f"\n[Max Threads] limit={gen.hw['max_threads_per_block']}")
all_thread_sp = set()
for c in gen.build_max_threads_constraints():
    ext_str = str(c['sym_extent'])
    sp_names = sorted(set(re.findall(r'sp_\d+_\d+', ext_str)))
    all_thread_sp.update(sp_names)
    val = eval_sym_extent(c['sym_extent'], sym.sym_map)
    print(f"  s{c['stage_id']}: {ext_str}")
    print(f"    SPs: {sp_names}, val={val}")

print(f"\n[Summary]")
print(f"  Shared memory SPs: {sorted(all_shared_sp)}")
print(f"  Max threads SPs: {sorted(all_thread_sp)}")
print(f"  Shared ∩ Thread: {sorted(all_shared_sp & all_thread_sp)}")

# SP 구조 — shared/thread에 참여하는 SP만 표시
print(f"\n[Relevant SP Groups]")
pm_t = SymParamManager(sym)
sp_groups = pm_t._build_sp_groups()
sp_extents = pm_t._build_sp_extents(sp_groups)
relevant = all_shared_sp | all_thread_sp
for step_idx in sorted(sp_groups.keys()):
    group = sp_groups[step_idx]
    if any(n in relevant for n in group):
        ext = sp_extents.get(step_idx, "?")
        vals = [sym.sym_map.get(n, "?") for n in group]
        tags = []
        for n in group:
            t = []
            if n in all_shared_sp: t.append("SM")
            if n in all_thread_sp: t.append("TH")
            tags.append("+".join(t) if t else "")
        print(f"  step={step_idx} extent={ext}: {group} = {vals}")
        print(f"    tags: {tags}")


Task 0: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_trans

[Shared Memory] limit=49152
  data_pack.shared: sp_10_0*sp_10_1*sp_10_2*sp_10_3*sp_11_0*sp_11_1*sp_11_2*sp_11_3*sp_12_0*sp_12_1*sp_12_2*sp_12_3*sp_14_0*sp_14_1
    SPs: ['sp_10_0', 'sp_10_1', 'sp_10_2', 'sp_10_3', 'sp_11_0', 'sp_11_1', 'sp_11_2', 'sp_11_3', 'sp_12_0', 'sp_12_1', 'sp_12_2', 'sp_12_3', 'sp_14_0', 'sp_14_1'], val=672, bytes=2688
  p1.shared: sp_10_0*sp_10_1*sp_10_2*sp_10_3*sp_11_0*sp_11_1*sp_11_2*sp_11_3*sp_13_0*sp_13_1*sp_13_2*sp_13_3*sp_14_0*sp_14_1
    SPs: ['sp_10_0', 'sp_10_1', 'sp_10_2', 'sp_10_3', 'sp_11_0', 'sp_11_1', 'sp_11_2', 'sp_11_3', 'sp_13_0', 'sp_13_1', 'sp_13_2', 'sp_13_3', 'sp_14_0', 'sp_14_1'], val=768, bytes=3072

[Max Threads] limit=1024
  s4: min(sp_61_0,ceil(196/(min(sp_28_0,196)))*ceil(64/(min(sp_29_0,64)))*min(sp_28_0,196)*min(sp_29_0,64))
    SPs: ['sp_28_0', 'sp_29_0', 'sp_61_0'], val=32
  s5: sp_13_1*sp_12_1*sp_11_1*sp_10_1
    SPs: ['sp_10_1', 'sp_11_1', 'sp_12_1', 'sp_13_1'

In [86]:
# ═══ 심층 분석: shared memory extent의 실제 SP 의존성 ═══
# shared_fused_extent가 SP 곱의 어떤 부분인지 정확히 분석

tidx = sorted(grouped.keys())[0]
task = tasks[tidx]
recs = grouped[tidx]
best = min(recs, key=lambda x: x[1].costs[0] if x[1].costs else float('inf'))
sym, st = create_sym_state_from_record(task, best[0])
gen = ScheduleGenerator(sym, HW_PARAM)

# shared_fused_extents 직접 확인
print("=== _shared_fused_extents ===")
for sid, ext in sorted(sym._shared_fused_extents.items()):
    print(f"  stage {sid} ({sym.stages[sid].op_name}): {ext}")

# split_sym_products 확인
print("\n=== _split_sym_products ===")
for (sid, step_idx), prod in sorted(sym._split_sym_products.items()):
    val = eval_sym_extent(prod, sym.sym_map)
    print(f"  stage={sid} step={step_idx}: {prod} = {val}")

# 핵심: shared extent가 SP 전체 곱이면 extent 값과 같다
# SP 그룹의 전체 곱 = extent (약수 조건에 의해)
# 따라서 shared memory는 SP 파라미터 선택에 무관하게 같을까?

# 검증: 여러 랜덤 파라미터로 shared extent 값 변화 확인
import random
rng = random.Random(42)
shared_vals = []
for trial in range(20):
    rng2 = random.Random(rng.randint(0, 10**9))
    pm_t2 = SymParamManager(sym)
    pm_t2.randomize_params(rng2)
    vals = []
    for item in gen.build_shared_memory_constraints()['items']:
        v = eval_sym_extent(item['sym_extent'], sym.sym_map)
        vals.append(v * item['dtype_bytes'])
    shared_vals.append(sum(vals))

print(f"\n=== Shared memory 총합 변화 (20 trials) ===")
print(f"  min={min(shared_vals)}, max={max(shared_vals)}, unique={len(set(shared_vals))}")
print(f"  values: {sorted(set(shared_vals))[:10]}...")
print(f"  limit: {gen.hw['max_shared_memory_per_block']}")


=== _shared_fused_extents ===
  stage 5 (data_pack.shared): sp_10_0*sp_10_1*sp_10_2*sp_10_3*sp_11_0*sp_11_1*sp_11_2*sp_11_3*sp_12_0*sp_12_1*sp_12_2*sp_12_3*sp_14_0*sp_14_1
  stage 7 (p1.shared): sp_10_0*sp_10_1*sp_10_2*sp_10_3*sp_11_0*sp_11_1*sp_11_2*sp_11_3*sp_13_0*sp_13_1*sp_13_2*sp_13_3*sp_14_0*sp_14_1

=== _split_sym_products ===
  stage=4 step=28: sp_28_0 = 7
  stage=4 step=29: sp_29_0 = 64
  stage=4 step=61: sp_61_0 = 32
  stage=5 step=56: sp_56_0 = 1
  stage=7 step=51: sp_51_0 = 4
  stage=8 step=10: sp_10_0*sp_10_1*sp_10_2*sp_10_3 = 1
  stage=8 step=11: sp_11_0*sp_11_1*sp_11_2*sp_11_3 = 6
  stage=8 step=12: sp_12_0*sp_12_1*sp_12_2*sp_12_3 = 28
  stage=8 step=13: sp_13_0*sp_13_1*sp_13_2*sp_13_3 = 32
  stage=8 step=14: sp_14_0*sp_14_1 = 4
  stage=11 step=3: sp_3_0 = 7
  stage=11 step=4: sp_4_0 = 32
  stage=11 step=41: sp_41_0 = 64
  stage=14 step=37: sp_37_0 = 32

=== Shared memory 총합 변화 (20 trials) ===
  min=37440, max=2396160, unique=11
  values: [37440, 149760, 299520, 599040, 

In [87]:
# ═══ 분석: shared memory 제약과 max_threads 제약의 SP 의존성 패턴 ═══
# 공통 패턴 발견을 위해 여러 task의 제약 구조 분석

import re
from collections import Counter

# 패턴: shared extent는 어떤 SP들의 곱인가?
# 패턴: max_threads extent는 어떤 SP들의 곱인가?

print("=== 전체 task의 제약 구조 요약 ===\n")

for tidx in sorted(grouped.keys()):
    task = tasks[tidx]
    recs = grouped[tidx]
    best = min(recs, key=lambda x: x[1].costs[0] if x[1].costs else float('inf'))
    sym_a, st_a = create_sym_state_from_record(task, best[0])
    gen_a = ScheduleGenerator(sym_a, HW_PARAM)

    # Shared memory items
    c_sh = gen_a.build_shared_memory_constraints()
    sh_count = len(c_sh['items'])

    # Max threads — FollowFusedSplit 곱 패턴 감지 (sp_X_1*sp_Y_1*...)
    th_constrs = gen_a.build_max_threads_constraints()
    ffs_threads = []  # 복합 곱 형태의 thread constraints
    simple_threads = []  # min(sp_X, N) 형태
    for c in th_constrs:
        ext_str = str(c['sym_extent'])
        # FollowFusedSplit 곱 패턴: sp_A_1*sp_B_1*sp_C_1*sp_D_1
        if re.match(r'^(sp_\d+_\d+\*?)+$', ext_str):
            ffs_threads.append(ext_str)
        else:
            simple_threads.append(ext_str)

    # 간결하게 출력
    pass_rate = per_task_stats.get(tidx, {}).get('rate', 1.0)
    marker = "  " if pass_rate == 1.0 else "⚠️"
    print(f"{marker} T{tidx:2d} (pass={100*pass_rate:.0f}%): "
          f"shared_stages={sh_count}, "
          f"thread_product={len(ffs_threads)}, "
          f"thread_simple={len(simple_threads)}")
    if ffs_threads:
        for ft in ffs_threads:
            print(f"      thread_product: {ft}")


=== 전체 task의 제약 구조 요약 ===

⚠️ T 0 (pass=5%): shared_stages=2, thread_product=2, thread_simple=4
      thread_product: sp_13_1*sp_12_1*sp_11_1*sp_10_1
      thread_product: sp_13_1*sp_12_1*sp_11_1*sp_10_1
⚠️ T 1 (pass=10%): shared_stages=2, thread_product=2, thread_simple=4
      thread_product: sp_13_1*sp_12_1*sp_11_1*sp_10_1
      thread_product: sp_13_1*sp_12_1*sp_11_1*sp_10_1
⚠️ T 2 (pass=8%): shared_stages=2, thread_product=2, thread_simple=4
      thread_product: sp_13_1*sp_12_1*sp_11_1*sp_10_1
      thread_product: sp_13_1*sp_12_1*sp_11_1*sp_10_1
⚠️ T 3 (pass=8%): shared_stages=2, thread_product=2, thread_simple=4
      thread_product: sp_13_1*sp_12_1*sp_11_1*sp_10_1
      thread_product: sp_13_1*sp_12_1*sp_11_1*sp_10_1
⚠️ T 4 (pass=7%): shared_stages=2, thread_product=2, thread_simple=4
      thread_product: sp_15_1*sp_14_1*sp_13_1*sp_12_1
      thread_product: sp_15_1*sp_14_1*sp_13_1*sp_12_1
⚠️ T 5 (pass=6%): shared_stages=2, thread_product=2, thread_simple=4
      thread_produ

In [90]:
# ═══ 신규 randomize_params 성능 테스트: rejection 비율 측정 ═══
import random
from collections import Counter

rng_base = random.Random(42)
n_trials = 500

rejection_stats = Counter()
total_attempts = 0
total_pass = 0
per_task_stats = {}

for tidx in sorted(grouped.keys()):
    task = tasks[tidx]
    recs = grouped[tidx]
    best = min(recs, key=lambda x: x[1].costs[0] if x[1].costs else float('inf'))

    sym, st = create_sym_state_from_record(task, best[0])
    gen = ScheduleGenerator(sym, HW_PARAM)

    task_total = 0
    task_pass_count = 0
    task_violations = Counter()

    for trial in range(n_trials):
        task_total += 1
        total_attempts += 1

        rng = random.Random(rng_base.randint(0, 10**9))
        try:
            gen.randomize_params(rng, max_retries=1)  # single attempt
            task_pass_count += 1
            total_pass += 1
        except RuntimeError:
            for name, checker in [
                ("vectorize", gen.check_vectorize),
                ("shared_memory", gen.check_shared_memory),
                ("max_threads", gen.check_max_threads),
                ("min_thread", gen.check_min_thread),
                ("vthread", gen.check_vthread),
                ("innermost", gen.check_innermost_split),
            ]:
                v = checker()
                if v:
                    task_violations[name] += 1
                    rejection_stats[name] += 1

    per_task_stats[tidx] = {
        'total': task_total, 'pass': task_pass_count,
        'rate': task_pass_count/task_total if task_total > 0 else 0,
        'violations': dict(task_violations),
    }

print(f"=== 신규 전략 통계: {total_pass}/{total_attempts} 통과 ({100*total_pass/total_attempts:.1f}%) ===")
print(f"(이전: 23.2%)")
if total_attempts > total_pass:
    print(f"\n위반 유형별 빈도:")
    for k, v in rejection_stats.most_common():
        pct = 100*v/(total_attempts - total_pass)
        print(f"  {k}: {v} ({pct:.1f}% of failures)")

print(f"\n=== Task별 상세 (실패 있는 것만) ===")
for tidx in sorted(per_task_stats.keys()):
    s = per_task_stats[tidx]
    if s['pass'] < s['total']:
        print(f"  T{tidx:2d}: {s['pass']}/{s['total']} ({100*s['rate']:.0f}%) violations={s['violations']}")


=== 신규 전략 통계: 10810/12000 통과 (90.1%) ===
(이전: 23.2%)

위반 유형별 빈도:
  shared_memory: 1190 (100.0% of failures)

=== Task별 상세 (실패 있는 것만) ===
  T13: 182/500 (36%) violations={'shared_memory': 318}
  T14: 292/500 (58%) violations={'shared_memory': 208}
  T15: 392/500 (78%) violations={'shared_memory': 108}
  T16: 120/500 (24%) violations={'shared_memory': 380}
  T17: 402/500 (80%) violations={'shared_memory': 98}
  T18: 435/500 (87%) violations={'shared_memory': 65}
  T19: 487/500 (97%) violations={'shared_memory': 13}


In [91]:
# ═══ 디버깅: shared memory 위반 사례의 SP 값 분석 ═══
# T16이 가장 낮은 통과율 — 상세 분석

tidx = 16
task = tasks[tidx]
recs = grouped[tidx]
best = min(recs, key=lambda x: x[1].costs[0] if x[1].costs else float('inf'))
sym_d, st_d = create_sym_state_from_record(task, best[0])
gen_d = ScheduleGenerator(sym_d, HW_PARAM)

# shared memory 구조 확인
c_sh = gen_d.build_shared_memory_constraints()
print(f"Task {tidx}: shared memory limit={c_sh['limit']}")
for item in c_sh['items']:
    ext_str = str(item['sym_extent'])
    sp_names = sorted(set(re.findall(r'sp_\\d+_\\d+', ext_str)))
    print(f"  {item['op_name']}: dtype_bytes={item['dtype_bytes']}")
    # step별로 그룹핑
    from collections import defaultdict
    step_groups = defaultdict(list)
    for n in sp_names:
        si = int(n.split('_')[1])
        step_groups[si].append(n)
    for si in sorted(step_groups.keys()):
        ext = gen_d.pm._build_sp_extents(gen_d.pm._build_sp_groups()).get(si, '?')
        print(f"    step={si} extent={ext}: {step_groups[si]}")

# 위반 시 SP값과 shared memory 계산 확인
print(f"\n위반 사례 분석 (5개):")
rng_d = random.Random(999)
fail_count = 0
for trial in range(200):
    rng2 = random.Random(rng_d.randint(0, 10**9))
    sym_d2, _ = create_sym_state_from_record(task, best[0])
    gen_d2 = ScheduleGenerator(sym_d2, HW_PARAM)
    try:
        gen_d2.randomize_params(rng2, max_retries=1)
    except RuntimeError:
        fail_count += 1
        if fail_count <= 5:
            # shared memory 실제 값 확인
            total_bytes = 0
            for item in gen_d2.build_shared_memory_constraints()['items']:
                val = eval_sym_extent(item['sym_extent'], sym_d2.sym_map)
                b = val * item['dtype_bytes']
                total_bytes += b
                print(f"  {item['op_name']}: extent={val}, bytes={b}")
            print(f"  TOTAL: {total_bytes} vs limit={c_sh['limit']}")

            # SP 그룹별 곱 확인
            sp_g = gen_d2.pm._build_sp_groups()
            sp_e = gen_d2.pm._build_sp_extents(sp_g)
            for si in sorted(sp_g.keys()):
                group = sp_g[si]
                vals = [sym_d2.sym_map[n] for n in group]
                prod = 1
                for v in vals:
                    prod *= v
                ext = sp_e.get(si, '?')
                print(f"    step={si} extent={ext}: {vals} prod={prod}")
            print()
print(f"Failures: {fail_count}/200")


Task 16: shared memory limit=49152
  pad_temp.shared: dtype_bytes=4
  p1.shared: dtype_bytes=4

위반 사례 분석 (5개):
  pad_temp.shared: extent=24753, bytes=99012
  p1.shared: extent=64, bytes=256
  TOTAL: 99268 vs limit=49152
    step=1 extent=1: [1, 1, 1, 1] prod=1
    step=2 extent=112: [2, 4, 14, 1] prod=112
    step=3 extent=112: [4, 7, 2, 1] prod=56
    step=4 extent=64: [1, 4, 16, 1] prod=64
    step=5 extent=7: [1, 1] prod=1
    step=6 extent=7: [1, 1] prod=1
    step=7 extent=3: [1, 1] prod=1
    step=27 extent=294: [3] prod=3
    step=32 extent=507: [1] prod=1

  pad_temp.shared: extent=36795, bytes=147180
  p1.shared: extent=96, bytes=384
  TOTAL: 147564 vs limit=49152
    step=1 extent=1: [1, 1, 1, 1] prod=1
    step=2 extent=112: [8, 2, 7, 1] prod=112
    step=3 extent=112: [1, 14, 2, 1] prod=28
    step=4 extent=64: [1, 4, 8, 1] prod=32
    step=5 extent=7: [1, 1] prod=1
    step=6 extent=7: [1, 1] prod=1
    step=7 extent=3: [3, 1] prod=3
    step=27 extent=294: [3] prod=3
    

In [92]:
# ═══ 핵심 디버깅: shared_fused_extent eval vs SP값 곱 ═══
# T16, 첫 번째 위반 사례에서 상세 확인

tidx = 16
task = tasks[tidx]
recs = grouped[tidx]
best = min(recs, key=lambda x: x[1].costs[0] if x[1].costs else float('inf'))

# 위반 사례 하나 생성
sym_d3, _ = create_sym_state_from_record(task, best[0])
gen_d3 = ScheduleGenerator(sym_d3, HW_PARAM)
rng3 = random.Random(12345)
try:
    gen_d3.randomize_params(rng3, max_retries=1)
except RuntimeError:
    pass

# shared_fused_extents 확인
print("=== shared_fused_extents ===")
for sid, ext in sorted(sym_d3._shared_fused_extents.items()):
    ext_str = str(ext)
    print(f"  stage {sid} ({sym_d3.stages[sid].op_name}):")
    print(f"    expr = {ext_str}")

    # eval_sym_extent 결과
    eval_val = eval_sym_extent(ext, sym_d3.sym_map)
    print(f"    eval = {eval_val}")

    # 순수 SP 값 곱 (식에서 SP 이름 추출하여 값의 곱 계산)
    sp_names_in = re.findall(r'sp_\d+_\d+', ext_str)
    pure_prod = 1
    for n in sp_names_in:
        v = sym_d3.sym_map.get(n, 1)
        pure_prod *= v
    print(f"    SP값 순수곱 = {pure_prod}")
    print(f"    차이 = {eval_val - pure_prod}")

# eval_sym_extent가 어떻게 작동하는지 — 식에 min/ceildiv가 있는지
print("\n=== 실제 shared stage의 fuse된 iters 확인 ===")
for sid, ext in sorted(sym_d3._shared_fused_extents.items()):
    stg = sym_d3.stages[sid]
    print(f"\nStage {sid} ({stg.op_name}) iters:")
    for iid, it in enumerate(stg.iters):
        print(f"  [{iid}] {it.name}: extent={it.extent}")


=== shared_fused_extents ===
  stage 2 (pad_temp.shared):
    expr = sp_1_0*sp_1_1*sp_1_2*sp_1_3*((sp_2_0*sp_2_1*sp_2_2*sp_2_3 - 1)*2 + sp_5_0*sp_5_1)*((sp_3_0*sp_3_1*sp_3_2*sp_3_3 - 1)*2 + sp_6_0*sp_6_1)*sp_7_0*sp_7_1
    eval = 24753
    SP값 순수곱 = 6272
    차이 = 18481
  stage 4 (p1.shared):
    expr = sp_5_0*sp_5_1*sp_6_0*sp_6_1*sp_7_0*sp_7_1*sp_4_0*sp_4_1*sp_4_2*sp_4_3
    eval = 32
    SP값 순수곱 = 32
    차이 = 0

=== 실제 shared stage의 fuse된 iters 확인 ===

Stage 2 (pad_temp.shared) iters:
  [0] ax0@ax1@ax2@ax3@.0.0: extent=ceil(ceil(sp_1_0*sp_1_1*sp_1_2*sp_1_3*((sp_2_0*sp_2_1*sp_2_2*sp_2_3 - 1)*2 + sp_5_0*sp_5_1)*((sp_3_0*sp_3_1*sp_3_2*sp_3_3 - 1)*2 + sp_6_0*sp_6_1)*sp_7_0*sp_7_1/(min(sp_32_0,sp_1_0*sp_1_1*sp_1_2*sp_1_3*((sp_2_0*sp_2_1*sp_2_2*sp_2_3 - 1)*2 + sp_5_0*sp_5_1)*((sp_3_0*sp_3_1*sp_3_2*sp_3_3 - 1)*2 + sp_6_0*sp_6_1)*sp_7_0*sp_7_1)))/(sp_4_1*sp_3_1*sp_2_1*sp_1_1))
  [1] ax0@ax1@ax2@ax3@.0.1: extent=sp_4_1*sp_3_1*sp_2_1*sp_1_1
  [2] ax0@ax1@ax2@ax3@.1: extent=min(sp_32_0,sp_1_0*sp

In [97]:
# ═══ ScheduleGenerator 최종 성능 테스트 (최적화: sym_state 재사용) ═══
import random, time
from collections import Counter

rng_base = random.Random(42)
n_trials = 500

total_ok = 0
total_fail = 0
fail_details = []
rejection_stats = Counter()
t0 = time.time()

for tidx in sorted(grouped.keys()):
    task = tasks[tidx]
    recs = grouped[tidx]
    best = min(recs, key=lambda x: x[1].costs[0] if x[1].costs else float('inf'))

    # sym_state를 한 번만 생성하고, gen을 재사용
    sym_t, _ = create_sym_state_from_record(task, best[0])
    gen_t = ScheduleGenerator(sym_t, HW_PARAM)

    task_ok = 0
    task_fail = 0
    for trial in range(n_trials):
        rng = random.Random(rng_base.randint(0, 10**9))

        try:
            gen_t.randomize_params(rng, max_retries=1)
            task_ok += 1
            total_ok += 1
        except RuntimeError:
            task_fail += 1
            total_fail += 1
            for name, checker in [
                ("vectorize", gen_t.check_vectorize),
                ("shared_memory", gen_t.check_shared_memory),
                ("max_threads", gen_t.check_max_threads),
                ("min_thread", gen_t.check_min_thread),
                ("vthread", gen_t.check_vthread),
                ("innermost", gen_t.check_innermost_split),
            ]:
                v = checker()
                if v:
                    rejection_stats[name] += 1
                    if total_fail <= 3:
                        fail_details.append(f"T{tidx}: {v}")

    if task_fail > 0:
        print(f"  T{tidx:2d}: {task_ok}/{n_trials} ({100*task_ok/n_trials:.0f}%)")

elapsed = time.time() - t0
total = total_ok + total_fail
print(f"\n=== 전체: {total_ok}/{total} 통과 ({100*total_ok/total:.1f}%) elapsed={elapsed:.1f}s ===")
if rejection_stats:
    print(f"\n위반 유형:")
    for k, v in rejection_stats.most_common():
        print(f"  {k}: {v}")
if fail_details:
    print(f"\n실패 상세:")
    for d in fail_details:
        print(f"  {d}")



=== 전체: 12000/12000 통과 (100.0%) elapsed=18.5s ===


In [ ]:
def group_population_by_task(search_policies, tasks):

    grouped = defaultdict(list)

    for tidx, policy in enumerate(search_policies):
        task = tasks[tidx]
        print(f"Sampling initial population for Task {tidx}: {task.desc}...")
        states = policy.sample_initial_population()
        measure_inputs = [MeasureInput(task, s) for s in states]
        measure_results = [MeasureResult([1e10], 0, "", 0.0, 0.0)] * len(measure_inputs)  # Placeholder for results
        grouped[tidx].extend(zip(measure_inputs, measure_results))
    
    return grouped